# A04 · DNN: tiểu đường, giá nhà và khách hàng

Notebook là nơi đọc và sửa toàn bộ implementation dưới đây: dữ liệu, mô hình, loss, optimizer, train/đánh giá. Không cần mở file `.py` để xem thuật toán. Các bản `.py` cũ giữ nguyên để bảo toàn backend/artifact.

**Cách đọc:** khởi tạo → code từng hàm/lớp → thực hành theo dataset → kết quả. `%%inline_module` chỉ chạy code có ngay trong cell ở namespace riêng, không nạp implementation từ `.py`. Mỗi framework dùng kernel WSL riêng; code của cả hai vẫn hiện trong notebook. Mặc định không train/TEST lại.

In [ ]:
PROJECT_NAME = 'A04'
from pathlib import Path
import sys, os, types, __future__
from IPython.core.magic import register_cell_magic
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ASSIGN_REORGANIZATION_EXECUTION_PLAN.md").exists())
sys.path.insert(0, str(REPO / "_shared"))
from assignment_runtime import activate
ACTIVE_FRAMEWORK = "keras" if ".venv-keras" in sys.executable else "torch"
_context = activate(PROJECT_NAME, ACTIVE_FRAMEWORK)
LOGICAL_ROOT = Path(_context["logical_root"])
RUN_EXPERIMENTS = False
RUN_TRAINING = False
RUN_TEST = False
# Kết quả của lượt chạy mới nằm ngoài notebooks/, không ghi vào artifact gốc.
import assignment_runtime as _paths
_paths._WRITE_CONTEXT["output"] = REPO / PROJECT_NAME / "artifacts/notebook_runs/inline_v1"
_paths._WRITE_CONTEXT["overlay"] = False
INLINE_MODULES = {}

def _notebook_package(name):
    if not name:
        return None
    if name not in INLINE_MODULES:
        parent, _, child = name.rpartition(".")
        package = _notebook_package(parent)
        module = types.ModuleType(name)
        module.__package__ = parent
        module.__path__ = []  # Không tìm implementation khác trong file .py.
        module.__file__ = str(LOGICAL_ROOT / (name.replace(".", "/") + "/__init__.py"))
        module.__dict__.update(ACTIVE_FRAMEWORK=ACTIVE_FRAMEWORK, RUN_TRAINING=False, RUN_TEST=False)
        INLINE_MODULES[name] = module
        sys.modules[name] = module
        if package is not None:
            setattr(package, child, module)
    return INLINE_MODULES[name]

@register_cell_magic
def inline_module(line, cell):
    """Chạy đúng code trong cell ở namespace riêng; không đọc code từ .py."""
    name, relative, scope = line.split()
    if scope != "both" and scope != ACTIVE_FRAMEWORK:
        return
    parent, _, child = name.rpartition(".")
    _notebook_package(parent)
    module = _notebook_package(name)
    module.__file__ = str(LOGICAL_ROOT / relative)
    module.__package__ = name if relative.endswith("/__init__.py") else parent
    compiled = compile(cell, "<notebook>/" + name, "exec", flags=__future__.annotations.compiler_flag, dont_inherit=True)
    exec(compiled, module.__dict__)
    module.__notebook_source__ = True

print({"bài": PROJECT_NAME, "kernel": ACTIVE_FRAMEWORK, "train": RUN_TRAINING, "TEST": RUN_TEST})


## Thực hành: customer_behavior_keras

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

# A04 customer_behavior — keras

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    RUN_TRAINING = False
    if RUN_TRAINING:
        from assignment_runtime import enable_training_output_overlay
        enable_training_output_overlay()
    from pathlib import Path
    import json
    import pandas as pd
    PROJECT_ROOT = Path(_migration['logical_root'])


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Customer Behavior: dự đoán customer churn từ hành vi mua hàng và tương tác của khách. A04 dùng Olist Brazilian E-Commerce Public Dataset đã được chuẩn bị thành customer-level churn dataset trong `datasets/customer_behavior/a04`.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        from pathlib import Path
        import json
        import pandas as pd

        def find_project_root(start=None):
            start = Path.cwd() if start is None else Path(start)
            for path in [start.resolve(), *start.resolve().parents]:
                if (path / "datasets" / "customer_behavior" / "a04").exists() and (path / "pipeline").exists():
                    return path
            raise FileNotFoundError("Cannot locate project root from current working directory.")

        PROJECT_ROOT = find_project_root()
        DATA_DIR = PROJECT_ROOT / "datasets" / "customer_behavior" / "a04"
        A03_NOTEBOOK = PROJECT_ROOT / "pipeline" / "customer_behavior_pipeline.ipynb"
        A03_METADATA_PATH = PROJECT_ROOT / "pipeline" / "customer_behavior" / "metadata.json"
        A03_COMPARISON_PATH = PROJECT_ROOT / "pipeline" / "customer_behavior" / "final_comparison.csv"
        A04_METADATA_PATH = DATA_DIR / "metadata.json"

        with A04_METADATA_PATH.open("r", encoding="utf-8") as f:
            dataset_metadata = json.load(f)
        with A03_METADATA_PATH.open("r", encoding="utf-8") as f:
            a03_metadata_preview = json.load(f)

        overview = {
            "A03 business problem": a03_metadata_preview["task_name"],
            "A03 target": a03_metadata_preview["target"],
            "A03 task type": a03_metadata_preview["task_type"],
            "A03 positive label": "is_churned = 1 means churned customer",
            "A04 dataset": dataset_metadata["dataset_name"],
            "A04 source": dataset_metadata["source"],
            "A04 target": dataset_metadata["target"],
            "A04 samples": dataset_metadata["samples"],
            "A04 features": dataset_metadata["features"],
            "Task type": "Binary classification",
        }
        pd.DataFrame([overview]).T.rename(columns={0: "Value"})


Output cho thấy điều gì?

Bảng overview xác nhận A04 giữ đúng business problem của A03: dự đoán `is_churned` cho khách hàng. Dataset A04 là Olist customer-level churn với 58,101 samples, 29 raw input features và target nhị phân; `is_churned=1` nghĩa là khách không có repeat order hợp lệ trong outcome window 180 ngày.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để tránh hard-code đường dẫn cá nhân.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    import os
    import random
    import time
    import warnings
    from datetime import datetime, timezone
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import sklearn
    from IPython.display import display
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    warnings.filterwarnings('ignore', category=FutureWarning)
    SEED = 42
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
    tf.get_logger().setLevel('ERROR')
    tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
    ARTIFACT_DIR = PROJECT_ROOT / 'pipeline04' / 'customer_behavior'
    MODEL_DIR = ARTIFACT_DIR / 'models'
    METRIC_DIR = ARTIFACT_DIR / 'metrics'
    FIGURE_DIR = ARTIFACT_DIR / 'figures'
    for directory in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'TensorFlow': tf.__version__, 'TensorFlow GPUs': len(tf.config.list_physical_devices('GPU')), 'Project root is portable': PROJECT_ROOT.name == 'assign'}
    pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


Output cho thấy điều gì?

Environment có đủ TensorFlow và PyTorch để chạy cùng một prepared customer dataset. CPU vẫn chạy được vì dữ liệu đã aggregate ở cấp customer, không phải event stream thô nhiều triệu dòng.


# 3. Load Dataset

Cell này làm gì?

Liệt kê raw Olist files và đọc prepared customer-level churn CSV. Raw files được giữ nguyên; notebook dùng file prepared để tránh tự tạo target khác A03.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        all_files = sorted(DATA_DIR.glob("*.csv"))
        file_listing = [
            {"file": path.relative_to(PROJECT_ROOT).as_posix(), "size_mb": round(path.stat().st_size / (1024 * 1024), 2)}
            for path in all_files
        ]
        display(pd.DataFrame(file_listing))

        DATA_PATH = DATA_DIR / "olist_customer_churn_a04.csv"
        TARGET = "is_churned"
        df_raw = pd.read_csv(DATA_PATH)

        print("Loaded dataset:", DATA_PATH.relative_to(PROJECT_ROOT).as_posix())
        print("Shape:", df_raw.shape)
        display(df_raw.head())


Output cho thấy điều gì?

Folder A04 có raw relational Olist tables và một prepared customer-level file. Notebook đọc `olist_customer_churn_a04.csv` vì file này giữ target `is_churned` đã chốt cho continuity với A03.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra shape, columns, target distribution, missing values, duplicates, descriptive statistics, categorical columns và evidence về sequence/time structure.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        shape_info = pd.DataFrame(
            {"Metric": ["Rows", "Columns", "Candidate feature columns"], "Value": [df_raw.shape[0], df_raw.shape[1], df_raw.shape[1] - 2]}
        )
        dtype_info = df_raw.dtypes.astype(str).reset_index()
        dtype_info.columns = ["Column", "Dtype"]
        missing = df_raw.isna().sum().reset_index()
        missing.columns = ["Column", "MissingCount"]
        missing["MissingRate"] = missing["MissingCount"] / len(df_raw)
        duplicate_count = int(df_raw.duplicated().sum())

        target_counts = df_raw[TARGET].value_counts().sort_index()
        target_distribution = pd.DataFrame(
            {
                "Class": target_counts.index.astype(int),
                "Count": target_counts.values,
                "Rate": (target_counts.values / len(df_raw)).round(4),
                "LabelInterpretation": ["not churned / returned in outcome window", "churned / no repeat order in outcome window"],
            }
        )

        categorical_preview = df_raw.select_dtypes(include=["object"]).nunique().reset_index()
        categorical_preview.columns = ["Column", "UniqueValues"]
        numeric_columns = [c for c in df_raw.select_dtypes(include=[np.number]).columns if c != TARGET]

        orders_path = DATA_DIR / "olist_orders_dataset.csv"
        orders_raw = pd.read_csv(orders_path, usecols=["order_id", "customer_id", "order_status", "order_purchase_timestamp"])
        orders_raw["order_purchase_timestamp"] = pd.to_datetime(orders_raw["order_purchase_timestamp"], errors="coerce")
        sequence_evidence = pd.DataFrame(
            [
                {"Representation": "prepared customer CSV", "Evidence": "one row per customer_unique_id with aggregate behavior features", "ModelingUse": "main tabular MLP input"},
                {"Representation": "raw Olist orders", "Evidence": f"{orders_raw.shape[0]:,} timestamped order events from {orders_raw['order_purchase_timestamp'].min()} to {orders_raw['order_purchase_timestamp'].max()}", "ModelingUse": "evidence of temporal source, not used as Conv1D input in this notebook"},
                {"Representation": "clickstream/session sequence", "Evidence": "not present in prepared file or raw Olist tables", "ModelingUse": "no CNN/sequence production model"},
            ]
        )

        display(shape_info)
        display(dtype_info)
        display(missing[missing["MissingCount"] > 0] if missing["MissingCount"].sum() else pd.DataFrame({"Missing": ["No missing values detected"]}))
        print("Duplicate rows:", duplicate_count)
        display(target_distribution)
        display(df_raw[numeric_columns].describe().T.round(3))
        display(categorical_preview)
        display(sequence_evidence)

        fig, ax = plt.subplots(figsize=(5.5, 3.5))
        ax.bar(target_distribution["Class"].astype(str), target_distribution["Count"], color=["#4C78A8", "#F58518"])
        ax.set_title("Olist Customer Churn Label Distribution")
        ax.set_xlabel("is_churned")
        ax.set_ylabel("Count")
        for i, row in target_distribution.iterrows():
            ax.text(i, row["Count"], f'{row["Rate"]:.1%}', ha="center", va="bottom")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "target_churn_distribution.png", dpi=160)
        plt.show()


Output cho thấy điều gì?

Prepared dataset có 58,101 dòng và 31 cột, không có missing value trong file đã chuẩn bị và không có duplicate row. Target cực kỳ mất cân bằng: class 0 có 665 khách (1.14%), class 1 có 57,436 khách (98.86%); vì vậy notebook báo thêm Macro F1 và Balanced Accuracy để tránh bị accuracy đánh lừa.


# 5. Data Preparation

Cell này làm gì?

Loại identifier, group rare categories theo train only, chia train/validation/test bằng stratify, rồi fit imputer/scaler/one-hot encoder chỉ trên training data.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        identifier_columns = ["customer_unique_id"]
        numeric_features = dataset_metadata["numerical_features"]
        categorical_features = dataset_metadata["categorical_features"]
        feature_cols = numeric_features + categorical_features

        df_model = df_raw.drop_duplicates().reset_index(drop=True)
        X = df_model[feature_cols].copy()
        y = df_model[TARGET].astype(int)

        X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
            X, y, test_size=0.30, random_state=SEED, stratify=y
        )
        X_valid_raw, X_test_raw, y_valid, y_test = train_test_split(
            X_temp_raw, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
        )

        top_category_limits = {
            "primary_category": 30,
            "primary_payment_type": 10,
            "customer_city": 80,
            "customer_state": 30,
        }
        category_maps = {}
        for col in categorical_features:
            top_values = X_train_raw[col].fillna("Unknown").value_counts().head(top_category_limits[col]).index.tolist()
            category_maps[col] = set(top_values)

        def apply_rare_category_grouping(frame):
            out = frame.copy()
            for col in categorical_features:
                values = out[col].fillna("Unknown").astype(str)
                out[col] = values.where(values.isin(category_maps[col]), "Other")
            return out

        X_train_raw = apply_rare_category_grouping(X_train_raw)
        X_valid_raw = apply_rare_category_grouping(X_valid_raw)
        X_test_raw = apply_rare_category_grouping(X_test_raw)

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="median")),
                            ("scaler", StandardScaler()),
                        ]
                    ),
                    numeric_features,
                ),
                (
                    "categorical",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="most_frequent")),
                            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                        ]
                    ),
                    categorical_features,
                ),
            ],
            sparse_threshold=0.0,
        )

        X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
        X_valid = preprocessor.transform(X_valid_raw).astype("float32")
        X_test = preprocessor.transform(X_test_raw).astype("float32")
        y_train_np = y_train.to_numpy(dtype="float32")
        y_valid_np = y_valid.to_numpy(dtype="float32")
        y_test_np = y_test.to_numpy(dtype="float32")
        transformed_feature_names = preprocessor.get_feature_names_out()

        joblib.dump({"preprocessor": preprocessor, "category_maps": {k: sorted(v) for k, v in category_maps.items()}}, MODEL_DIR / "preprocessor.joblib")

        split_summary = pd.DataFrame(
            [
                {"Split": "train", "Rows": len(y_train), "ChurnRate": y_train.mean(), "NotChurned": int((y_train == 0).sum()), "Churned": int((y_train == 1).sum())},
                {"Split": "validation", "Rows": len(y_valid), "ChurnRate": y_valid.mean(), "NotChurned": int((y_valid == 0).sum()), "Churned": int((y_valid == 1).sum())},
                {"Split": "test", "Rows": len(y_test), "ChurnRate": y_test.mean(), "NotChurned": int((y_test == 0).sum()), "Churned": int((y_test == 1).sum())},
            ]
        )
        preprocess_summary = pd.DataFrame(
            [{"RawFeatures": len(feature_cols), "NumericFeatures": len(numeric_features), "CategoricalFeatures": len(categorical_features), "TransformedFeatures": X_train.shape[1]}]
        )

        display(split_summary.round(4))
        display(preprocess_summary)
        display(pd.DataFrame({"TransformedFeature": transformed_feature_names[:25]}))


Output cho thấy điều gì?

Split được stratify thành train/validation/test và preprocessor chỉ fit trên train set. Sau encoding và scaling, mỗi khách hàng được biểu diễn bằng vector tabular gồm 169 features; target test vẫn được giữ riêng cho final evaluation.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của tensor đầu vào deep learning và đánh giá representation có phù hợp với CNN/sequence modeling không.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        representation_summary = pd.DataFrame(
            {
                "Array": ["X_train", "y_train", "X_valid", "y_valid", "X_test", "y_test"],
                "Shape": [X_train.shape, y_train_np.shape, X_valid.shape, y_valid_np.shape, X_test.shape, y_test_np.shape],
                "Meaning": [
                    "(samples, transformed tabular features)",
                    "(samples,)",
                    "(samples, transformed tabular features)",
                    "(samples,)",
                    "(samples, transformed tabular features)",
                    "(samples,)",
                ],
            }
        )
        representation_decision = pd.DataFrame(
            [
                {"Question": "Independent tabular features?", "Assessment": "Yes: prepared file is one customer row with aggregate behavioral features"},
                {"Question": "Ordered session/clickstream?", "Assessment": "No: prepared CSV has no event order or clickstream columns"},
                {"Question": "Raw transaction sequence exists?", "Assessment": "Partially: raw Olist order timestamps exist, but the model input selected here is customer-level aggregate"},
                {"Question": "Natural baseline", "Assessment": "MLP/DNN classification"},
                {"Question": "CNN justified?", "Assessment": "No production CNN for prepared representation; only toy Conv1D operation is shown"},
            ]
        )
        display(representation_summary)
        display(representation_decision)


Output cho thấy điều gì?

Representation chính là customer-level aggregate tabular vector, không phải ảnh và cũng chưa phải event sequence. Raw Olist có timestamped orders/payments/reviews, nhưng notebook này không tự reorder columns để tạo sequence giả; MLP/DNN là baseline tự nhiên, còn Conv1D chỉ được minh họa như operation giáo dục.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa Dense/Linear, ReLU, logits, sigmoid, Conv1D và pooling trên toy tensors.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        toy_x = tf.constant([[2.0, 0.5, -1.0]], dtype=tf.float32)
        dense = keras.Sequential(
            [
                layers.Input(shape=(3,)),
                layers.Dense(2, use_bias=True, kernel_initializer="ones", bias_initializer="zeros"),
            ]
        )
        toy_logits = dense(toy_x)
        toy_relu = tf.nn.relu(toy_logits)
        toy_sigmoid = tf.nn.sigmoid(toy_logits)

        toy_seq = tf.reshape(tf.range(1, 9, dtype=tf.float32), (1, 8, 1))
        conv = layers.Conv1D(filters=2, kernel_size=3, padding="valid", activation="relu")
        conv_out = conv(toy_seq)
        pool_out = layers.MaxPooling1D(pool_size=2)(conv_out)

        operation_shapes = pd.DataFrame(
            [
                {"Object": "toy_x", "Shape": tuple(toy_x.shape), "Meaning": "one toy customer vector"},
                {"Object": "Dense logits", "Shape": tuple(toy_logits.shape), "Meaning": "linear outputs before probability"},
                {"Object": "ReLU(logits)", "Shape": tuple(toy_relu.shape), "Meaning": "negative values clipped to zero"},
                {"Object": "Sigmoid(logits)", "Shape": tuple(toy_sigmoid.shape), "Meaning": "probability-like values 0..1"},
                {"Object": "Toy sequence", "Shape": tuple(toy_seq.shape), "Meaning": "(batch, steps, channels)"},
                {"Object": "Conv1D output", "Shape": tuple(conv_out.shape), "Meaning": "local windows over ordered toy sequence"},
                {"Object": "Pooling output", "Shape": tuple(pool_out.shape), "Meaning": "downsampled Conv1D features"},
            ]
        )
        display(operation_shapes)
        print("Dense logits:", toy_logits.numpy().round(4))
        print("Sigmoid probabilities:", toy_sigmoid.numpy().round(4))


Output cho thấy điều gì?

Dense tạo logits từ vector customer features, sigmoid chuyển logits thành xác suất churn. Conv1D/pooling chỉ minh họa trên toy sequence có thứ tự rõ, không áp đặt lên các cột aggregate của Olist.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, sigmoid, binary cross entropy, forward composition và Conv1D toy.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def linear_np(x, w, b):
            return x @ w + b

        def relu_np(x):
            return np.maximum(0, x)

        def sigmoid_np(x):
            return 1.0 / (1.0 + np.exp(-x))

        def binary_cross_entropy_np(y_true, y_prob, eps=1e-7):
            y_prob = np.clip(y_prob, eps, 1 - eps)
            return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

        def forward_np(x, params):
            z1 = linear_np(x, params["w1"], params["b1"])
            a1 = relu_np(z1)
            z2 = linear_np(a1, params["w2"], params["b2"])
            return sigmoid_np(z2)

        def conv1d_valid_np(sequence, kernel, bias=0.0):
            sequence = np.asarray(sequence, dtype=float)
            kernel = np.asarray(kernel, dtype=float)
            out = []
            for i in range(len(sequence) - len(kernel) + 1):
                out.append(float(np.sum(sequence[i : i + len(kernel)] * kernel) + bias))
            return np.asarray(out)

        rng = np.random.default_rng(SEED)
        toy_batch = X_train[:4]
        params = {
            "w1": rng.normal(0, 0.1, size=(X_train.shape[1], 8)),
            "b1": np.zeros(8),
            "w2": rng.normal(0, 0.1, size=(8, 1)),
            "b2": np.zeros(1),
        }
        toy_probs = forward_np(toy_batch, params).ravel()
        toy_loss = binary_cross_entropy_np(y_train_np[:4], toy_probs)
        toy_conv = conv1d_valid_np([1, 2, 3, 4, 5], [0.25, 0.5, 0.25])

        print("Forward churn probabilities:", np.round(toy_probs, 4))
        print("Toy BCE:", round(float(toy_loss), 4))
        print("Manual Conv1D toy output:", np.round(toy_conv, 4))


Output cho thấy điều gì?

NumPy forward pass minh họa cách MLP tạo xác suất churn từ customer aggregate features. Conv1D thủ công chỉ giải thích operation trên toy sequence, không dùng làm model chính cho prepared Olist table.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric, threshold selection và baseline Keras MLP cho churn classification.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def to_jsonable(obj):
            if isinstance(obj, dict):
                return {str(k): to_jsonable(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [to_jsonable(v) for v in obj]
            if isinstance(obj, tuple):
                return [to_jsonable(v) for v in obj]
            if isinstance(obj, (np.integer,)):
                return int(obj)
            if isinstance(obj, (np.floating,)):
                return float(obj)
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            return obj

        def save_json(path, obj):
            with Path(path).open("w", encoding="utf-8") as f:
                json.dump(to_jsonable(obj), f, indent=2)

        def select_threshold(y_true, prob):
            thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
            rows = []
            for threshold in thresholds:
                pred = (prob >= threshold).astype(int)
                rows.append(
                    {
                        "Threshold": float(threshold),
                        "Accuracy": accuracy_score(y_true, pred),
                        "Precision": precision_score(y_true, pred, zero_division=0),
                        "Recall": recall_score(y_true, pred, zero_division=0),
                        "F1": f1_score(y_true, pred, zero_division=0),
                        "MacroF1": f1_score(y_true, pred, average="macro", zero_division=0),
                        "BalancedAccuracy": balanced_accuracy_score(y_true, pred),
                    }
                )
            table = pd.DataFrame(rows)
            best = table.sort_values(["MacroF1", "BalancedAccuracy", "F1"], ascending=False).iloc[0]
            return float(best["Threshold"]), table

        def evaluate_probabilities(y_true, prob, threshold=None):
            if threshold is None:
                threshold, _ = select_threshold(y_true, prob)
            pred = (prob >= threshold).astype(int)
            cm = confusion_matrix(y_true, pred)
            tn, fp, fn, tp = cm.ravel()
            return {
                "Threshold": float(threshold),
                "Accuracy": float(accuracy_score(y_true, pred)),
                "Precision": float(precision_score(y_true, pred, zero_division=0)),
                "Recall": float(recall_score(y_true, pred, zero_division=0)),
                "F1": float(f1_score(y_true, pred, zero_division=0)),
                "MacroF1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
                "BalancedAccuracy": float(balanced_accuracy_score(y_true, pred)),
                "ROC-AUC": float(roc_auc_score(y_true, prob)),
                "TN": int(tn),
                "FP": int(fp),
                "FN": int(fn),
                "TP": int(tp),
            }

        def build_basic_mlp(input_dim, name="M1_Basic_MLP"):
            inputs = keras.Input(shape=(input_dim,), name="customer_behavior_vector")
            x = layers.Dense(128, activation="relu", name="dense_128")(inputs)
            x = layers.Dense(64, activation="relu", name="dense_64")(x)
            outputs = layers.Dense(1, activation="sigmoid", name="churn_probability")(x)
            model = keras.Model(inputs, outputs, name=name)
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                loss="binary_crossentropy",
                metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.AUC(name="auc")],
            )
            return model

        BATCH_SIZE = 2048
        MAX_EPOCHS = 12
        PATIENCE = 2
        input_dim = X_train.shape[1]

        keras_baseline = build_basic_mlp(input_dim)
        keras_baseline.summary()
        print("Input shape:", X_train.shape)
        print("Output shape for 5 samples:", tuple(keras_baseline(X_train[:5]).shape))
        print("Total parameters:", keras_baseline.count_params())


Output cho thấy điều gì?

Keras baseline nhận vector customer behavior đã transform và trả về xác suất `is_churned=1`. Vì label churn là class đa số, notebook chọn threshold theo Macro F1 để buộc model quan tâm cả retained/non-churn minority class.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, class weight, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def class_weight_from_y(y_array):
            neg = float((y_array == 0).sum())
            pos = float((y_array == 1).sum())
            total = neg + pos
            return {0: total / (2.0 * neg), 1: total / (2.0 * pos)}

        CLASS_WEIGHT = class_weight_from_y(y_train_np)

        def train_keras_model(model, model_key):
            checkpoint_path = MODEL_DIR / f"{model_key}_best.keras"
            callbacks = [
                keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE, restore_best_weights=True),
                keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_auc", mode="max", save_best_only=True),
            ]
            start = time.perf_counter()
            history = model.fit(
                X_train,
                y_train_np,
                validation_data=(X_valid, y_valid_np),
                epochs=MAX_EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=callbacks,
                class_weight=CLASS_WEIGHT,
                verbose=2,
            )
            train_time = time.perf_counter() - start
            history_df = pd.DataFrame(history.history)
            history_df.to_csv(METRIC_DIR / f"{model_key}_history.csv", index=False)
            return history, train_time, checkpoint_path

        keras_history, keras_train_time, keras_checkpoint = train_keras_model(keras_baseline, "m1_basic_mlp")
        print(f"Keras baseline train time: {keras_train_time:.2f} seconds")
        print("Best checkpoint:", keras_checkpoint.relative_to(PROJECT_ROOT).as_posix())
        display(pd.DataFrame(keras_history.history).tail())


Output cho thấy điều gì?

Training log cho thấy model học trên train và validation qua từng epoch. Class weight làm class 0 hiếm hơn có ảnh hưởng lớn hơn trong loss, còn test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và AUC của baseline Keras trên train/validation, đồng thời lưu figure.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def plot_history(history, title, output_path):
            history_df = pd.DataFrame(history.history)
            fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
            axes[0].plot(history_df["loss"], label="train")
            axes[0].plot(history_df["val_loss"], label="validation")
            axes[0].set_title(f"{title} Loss")
            axes[0].set_xlabel("Epoch")
            axes[0].legend()
            axes[1].plot(history_df["auc"], label="train")
            axes[1].plot(history_df["val_auc"], label="validation")
            axes[1].set_title(f"{title} AUC")
            axes[1].set_xlabel("Epoch")
            axes[1].legend()
            fig.tight_layout()
            fig.savefig(output_path, dpi=160)
            plt.show()

        plot_history(keras_history, "Keras Basic MLP", FIGURE_DIR / "keras_basic_mlp_training_curves.png")


Output cho thấy điều gì?

Curves giúp nhìn model có học tín hiệu churn beyond class prior hay không. Với imbalance lớn, validation AUC hữu ích vì không phụ thuộc vào một threshold cố định.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng Accuracy, Precision, Recall, F1, Macro F1, Balanced Accuracy, ROC-AUC, confusion matrix và classification report.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        keras_val_prob = keras_baseline.predict(X_valid, batch_size=BATCH_SIZE, verbose=0).ravel()
        keras_threshold, keras_threshold_table = select_threshold(y_valid_np, keras_val_prob)
        keras_metrics = evaluate_probabilities(y_valid_np, keras_val_prob, keras_threshold)
        keras_report = classification_report(
            y_valid_np.astype(int),
            (keras_val_prob >= keras_threshold).astype(int),
            output_dict=True,
            zero_division=0,
        )

        keras_result = {
            "Model": "M1 Basic MLP",
            "Framework": "Keras",
            **keras_metrics,
            "Params": int(keras_baseline.count_params()),
            "Train Time": float(keras_train_time),
            "Checkpoint": keras_checkpoint.relative_to(PROJECT_ROOT).as_posix(),
        }
        save_json(METRIC_DIR / "keras_basic_mlp_validation_metrics.json", keras_result)
        keras_threshold_table.to_csv(METRIC_DIR / "keras_basic_mlp_thresholds.csv", index=False)
        pd.DataFrame(keras_report).T.to_csv(METRIC_DIR / "keras_basic_mlp_classification_report.csv")

        display(pd.DataFrame([keras_result]).drop(columns=["Checkpoint"]))
        display(pd.DataFrame(confusion_matrix(y_valid_np, (keras_val_prob >= keras_threshold).astype(int)), index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"]))
        display(pd.DataFrame(keras_report).T.round(4))


Output cho thấy điều gì?

Keras baseline MLP dùng cùng representation tabular và threshold chọn trên validation. Baseline đạt validation F1 0.9916, Macro F1 0.5281, ROC-AUC 0.5915; positive-class F1 rất cao chủ yếu vì churn là class áp đảo, nên Macro F1 mới là tín hiệu quan trọng hơn.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước train, chạy một batch qua model để kiểm tra input/logits shape.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def count_torch_params(model):
            return sum((p.numel() for p in model.parameters() if p.requires_grad))


Output cho thấy điều gì?

Batch input có dạng `(batch, transformed features)` và logits có dạng `(batch, 1)`, đúng với binary churn classification. PyTorch model giữ cùng ý tưởng hidden layers như Keras baseline.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline với loop thủ công, weighted BCE cho imbalance, và lưu checkpoint tốt nhất theo validation ROC-AUC.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        torch_history_rows = []
        best_state = None
        epochs_without_improvement = 0


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ `zero_grad`, forward, weighted BCE, backward và optimizer step. Vì class 0 rất hiếm, weighted loss giúp model không chỉ tối ưu cho class churn đa số.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng protocol như Keras.


Output cho thấy điều gì?

PyTorch được đánh giá trên cùng validation set và cùng threshold rule. Với Olist, retained/non-churn là minority, nên classification report theo từng class cho biết model có bỏ qua class hiếm hay không.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng validation metrics.


Output cho thấy điều gì?

Keras và PyTorch dùng kiến trúc baseline tương đương về ý tưởng nhưng runtime khác nhau trên CPU. PyTorch baseline đạt validation Macro F1 0.5308, Keras baseline đạt 0.5281; đây chỉ là một run thực nghiệm, không phải kết luận framework nào tốt hơn về bản chất.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture MLP phù hợp dữ liệu tabular: M1 Basic MLP, M2 BatchNorm, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được train làm model chính vì input là aggregate tabular.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def compile_binary_model(model):
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                loss="binary_crossentropy",
                metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.AUC(name="auc")],
            )
            return model

        def build_batchnorm_mlp(input_dim, name="M2_BatchNorm_MLP"):
            inputs = keras.Input(shape=(input_dim,), name="customer_behavior_vector")
            x = layers.Dense(128, use_bias=False, name="dense_128")(inputs)
            x = layers.BatchNormalization(name="bn_128")(x)
            x = layers.Activation("relu", name="relu_128")(x)
            x = layers.Dense(64, use_bias=False, name="dense_64")(x)
            x = layers.BatchNormalization(name="bn_64")(x)
            x = layers.Activation("relu", name="relu_64")(x)
            outputs = layers.Dense(1, activation="sigmoid", name="churn_probability")(x)
            return compile_binary_model(keras.Model(inputs, outputs, name=name))

        def residual_block(x, units, block_name):
            shortcut = x
            y = layers.Dense(units, activation="relu", name=f"{block_name}_dense_a")(x)
            y = layers.Dense(units, name=f"{block_name}_dense_b")(y)
            x = layers.Add(name=f"{block_name}_add")([shortcut, y])
            return layers.Activation("relu", name=f"{block_name}_relu")(x)

        def build_residual_mlp(input_dim, name="M3_Residual_MLP"):
            inputs = keras.Input(shape=(input_dim,), name="customer_behavior_vector")
            x = layers.Dense(128, activation="relu", name="input_projection")(inputs)
            x = residual_block(x, 128, "residual_1")
            x = residual_block(x, 128, "residual_2")
            x = layers.Dense(64, activation="relu", name="dense_64")(x)
            outputs = layers.Dense(1, activation="sigmoid", name="churn_probability")(x)
            return compile_binary_model(keras.Model(inputs, outputs, name=name))

        def build_attention_residual_mlp(input_dim, name="M4_Residual_MLP_Feature_Attention"):
            inputs = keras.Input(shape=(input_dim,), name="customer_behavior_vector")
            attention = layers.Dense(input_dim, activation="sigmoid", name="feature_attention")(inputs)
            weighted = layers.Multiply(name="attention_weighted_features")([inputs, attention])
            x = layers.Dense(128, activation="relu", name="input_projection")(weighted)
            x = residual_block(x, 128, "residual_1")
            x = residual_block(x, 128, "residual_2")
            x = layers.Dense(64, activation="relu", name="dense_64")(x)
            outputs = layers.Dense(1, activation="sigmoid", name="churn_probability")(x)
            return compile_binary_model(keras.Model(inputs, outputs, name=name))

        def evaluate_keras_model(model, model_key, checkpoint_path, train_time):
            val_prob = model.predict(X_valid, batch_size=BATCH_SIZE, verbose=0).ravel()
            threshold, threshold_table = select_threshold(y_valid_np, val_prob)
            metrics = evaluate_probabilities(y_valid_np, val_prob, threshold)
            result = {
                "Model": model.name.replace("_", " "),
                "Framework": "Keras",
                **metrics,
                "Params": int(model.count_params()),
                "Train Time": float(train_time),
                "Checkpoint": checkpoint_path.relative_to(PROJECT_ROOT).as_posix(),
            }
            save_json(METRIC_DIR / f"{model_key}_validation_metrics.json", result)
            threshold_table.to_csv(METRIC_DIR / f"{model_key}_thresholds.csv", index=False)
            return result

        architecture_results = [keras_result]
        architecture_histories = {"M1 Basic MLP": keras_history}
        trained_keras_models = {"M1 Basic MLP": keras_baseline}

        for model_key, builder in [
            ("m2_batchnorm_mlp", build_batchnorm_mlp),
            ("m3_residual_mlp", build_residual_mlp),
            ("m4_attention_residual_mlp", build_attention_residual_mlp),
        ]:
            tf.random.set_seed(SEED)
            model = builder(input_dim)
            print("\nTraining", model.name, "with", model.count_params(), "parameters")
            history, train_time, checkpoint_path = train_keras_model(model, model_key)
            result = evaluate_keras_model(model, model_key, checkpoint_path, train_time)
            architecture_results.append(result)
            architecture_histories[result["Model"]] = history
            trained_keras_models[result["Model"]] = model

        cnn_investigation = pd.DataFrame(
            [
                {
                    "Investigation": "CNN on prepared customer aggregate features",
                    "Decision": "Not trained as a candidate model",
                    "Reason": "Columns are numeric/categorical aggregate variables, not ordered sessions or clickstream events.",
                }
            ]
        )
        display(cnn_investigation)


Output cho thấy điều gì?

Các experiment giữ cùng split, preprocessing, threshold rule, seed và training budget. CNN được dừng ở mức investigation vì prepared Olist input không còn event-order structure để Conv1D khai thác.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation metrics và chọn ứng viên tốt nhất bằng validation Macro F1.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        architecture_comparison = pd.DataFrame(architecture_results)[
            ["Model", "Accuracy", "Precision", "Recall", "F1", "MacroF1", "BalancedAccuracy", "ROC-AUC", "Params", "Train Time", "Threshold", "Checkpoint"]
        ].sort_values("MacroF1", ascending=False)
        architecture_comparison.to_csv(METRIC_DIR / "architecture_comparison.csv", index=False)

        display(architecture_comparison.drop(columns=["Checkpoint"]).round(4))

        fig, ax = plt.subplots(figsize=(8, 3.8))
        plot_df = architecture_comparison.sort_values("MacroF1")
        ax.barh(plot_df["Model"], plot_df["MacroF1"], color="#4C78A8")
        ax.set_xlabel("Validation Macro F1")
        ax.set_title("A04 Customer Behavior Architecture Comparison")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "architecture_comparison_macro_f1.png", dpi=160)
        plt.show()

        best_validation_row = architecture_comparison.iloc[0].to_dict()
        print("Best validation model:", best_validation_row["Model"])
        print("Best validation Macro F1:", round(best_validation_row["MacroF1"], 4))


Output cho thấy điều gì?

Trong nhóm architecture experiments, model tốt nhất theo validation Macro F1 là `M4 Residual MLP Feature Attention` với Macro F1 0.5293, F1 0.9919, ROC-AUC 0.5907. Chênh lệch giữa các MLP variant nhỏ, nên lợi ích thực tế của kiến trúc phức tạp cần được đọc cùng runtime và imbalance của target.


# 19. Representation Inspection

Cell này làm gì?

Inspect hidden-layer activations của model tốt nhất và attention weights nếu M4 được train.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        best_model_name = best_validation_row["Model"]
        best_checkpoint_path = PROJECT_ROOT / best_validation_row["Checkpoint"]
        best_validation_model = keras.models.load_model(best_checkpoint_path)

        hidden_layer = None
        for layer in best_validation_model.layers:
            if isinstance(layer, layers.Dense) and getattr(layer, "units", None) != 1:
                hidden_layer = layer.name
                break

        if hidden_layer is not None:
            activation_model = keras.Model(best_validation_model.input, best_validation_model.get_layer(hidden_layer).output)
            sample_activations = activation_model.predict(X_valid[:1024], batch_size=1024, verbose=0)
            activation_summary = pd.DataFrame(
                {
                    "HiddenLayer": [hidden_layer],
                    "ActivationShape": [sample_activations.shape],
                    "MeanActivation": [float(sample_activations.mean())],
                    "StdActivation": [float(sample_activations.std())],
                    "ZeroRate": [float((np.abs(sample_activations) < 1e-8).mean())],
                }
            )
            display(activation_summary.round(4))

        attention_model_path = MODEL_DIR / "m4_attention_residual_mlp_best.keras"
        if attention_model_path.exists():
            attention_model = keras.models.load_model(attention_model_path)
            attention_extractor = keras.Model(attention_model.input, attention_model.get_layer("feature_attention").output)
            attention_values = attention_extractor.predict(X_valid[:4096], batch_size=2048, verbose=0)
            attention_summary = pd.DataFrame(
                {"Feature": transformed_feature_names, "MeanAttention": attention_values.mean(axis=0), "StdAttention": attention_values.std(axis=0)}
            ).sort_values("MeanAttention", ascending=False)
            attention_summary.to_csv(METRIC_DIR / "feature_attention_summary.csv", index=False)
            display(attention_summary.head(12).round(4))
            fig, ax = plt.subplots(figsize=(8, 5))
            top_attention = attention_summary.head(12).sort_values("MeanAttention")
            ax.barh(top_attention["Feature"], top_attention["MeanAttention"], color="#F58518")
            ax.set_title("Top Feature Attention Weights")
            ax.set_xlabel("Mean attention")
            fig.tight_layout()
            fig.savefig(FIGURE_DIR / "feature_attention_weights.png", dpi=160)
            plt.show()


Output cho thấy điều gì?

Hidden activations cho biết model tạo latent representation từ behavior aggregates trước output churn. Attention weights, nếu xem từ M4, là tín hiệu model-inspection trên feature đã transform, không phải nguyên nhân kinh doanh chắc chắn.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi validation: confusion matrix, false positives, false negatives và high-confidence errors của model được chọn.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        best_val_prob = best_validation_model.predict(X_valid, batch_size=BATCH_SIZE, verbose=0).ravel()
        best_threshold = float(best_validation_row["Threshold"])
        best_val_pred = (best_val_prob >= best_threshold).astype(int)

        error_df = X_valid_raw.reset_index(drop=True).copy()
        error_df["y_true"] = y_valid_np.astype(int)
        error_df["prob_churn"] = best_val_prob
        error_df["y_pred"] = best_val_pred
        error_df["confidence"] = np.where(best_val_pred == 1, best_val_prob, 1 - best_val_prob)
        error_df["error_type"] = np.select(
            [
                (error_df["y_true"] == 0) & (error_df["y_pred"] == 1),
                (error_df["y_true"] == 1) & (error_df["y_pred"] == 0),
            ],
            ["False Positive", "False Negative"],
            default="Correct",
        )

        error_summary = error_df["error_type"].value_counts().rename_axis("ErrorType").reset_index(name="Count")
        high_conf_errors = error_df[error_df["error_type"] != "Correct"].sort_values("confidence", ascending=False).head(12)
        fp_fn_profile = error_df[error_df["error_type"].isin(["False Positive", "False Negative"])].groupby("error_type")[numeric_features].mean().T

        error_summary.to_csv(METRIC_DIR / "validation_error_summary.csv", index=False)
        high_conf_errors.to_csv(METRIC_DIR / "validation_high_confidence_errors.csv", index=False)
        fp_fn_profile.to_csv(METRIC_DIR / "validation_fp_fn_feature_means.csv")

        display(pd.DataFrame(confusion_matrix(y_valid_np, best_val_pred), index=["Actual retained", "Actual churned"], columns=["Pred retained", "Pred churned"]))
        display(error_summary)
        display(high_conf_errors[["y_true", "y_pred", "prob_churn", "confidence", "error_type", "days_since_last_order", "order_count", "avg_order_value", "avg_review_score", "primary_category", "primary_payment_type", "customer_state"]])
        display(fp_fn_profile.loc[["days_since_last_order", "order_count", "avg_order_value", "avg_review_score", "low_review_share", "on_time_delivery_rate"]].round(3))


Output cho thấy điều gì?

Error analysis trên validation có 8,575 dự đoán đúng, 95 false positives và 45 false negatives. Với churn, false negative nghĩa là model bỏ sót khách có khả năng churn; tuy nhiên do class churn chiếm gần toàn bộ dataset, cần thận trọng khi diễn giải độ tin cậy của từng nhóm lỗi.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation Macro F1, evaluate test set đúng một lần, rồi lưu final model.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        final_model = keras.models.load_model(best_checkpoint_path)
        test_prob = final_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
        final_threshold = float(best_validation_row["Threshold"])
        final_metrics = evaluate_probabilities(y_test_np, test_prob, final_threshold)
        final_pred = (test_prob >= final_threshold).astype(int)
        final_report = classification_report(y_test_np.astype(int), final_pred, output_dict=True, zero_division=0)

        final_model_path = MODEL_DIR / "final_model.keras"
        final_model.save(final_model_path)

        final_payload = {
            "selected_model": best_validation_row["Model"],
            "selection_metric": "Validation Macro F1",
            "decision_threshold": final_threshold,
            "test_metrics": final_metrics,
            "final_model_path": final_model_path.relative_to(PROJECT_ROOT).as_posix(),
            "source_checkpoint": best_checkpoint_path.relative_to(PROJECT_ROOT).as_posix(),
        }
        save_json(METRIC_DIR / "final_test_metrics.json", final_payload)
        pd.DataFrame(final_report).T.to_csv(METRIC_DIR / "final_classification_report.csv")

        final_cm = confusion_matrix(y_test_np, final_pred)
        fig, ax = plt.subplots(figsize=(4.5, 4))
        im = ax.imshow(final_cm, cmap="Blues")
        ax.set_xticks([0, 1], labels=["Pred retained", "Pred churned"], rotation=20)
        ax.set_yticks([0, 1], labels=["Actual retained", "Actual churned"])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(final_cm[i, j]), ha="center", va="center", color="black")
        ax.set_title("Final Test Confusion Matrix")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "final_confusion_matrix.png", dpi=160)
        plt.show()

        fpr, tpr, _ = roc_curve(y_test_np, test_prob)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(fpr, tpr, label=f"ROC-AUC={final_metrics['ROC-AUC']:.4f}")
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("Final Test ROC Curve")
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "final_roc_curve.png", dpi=160)
        plt.show()

        display(pd.DataFrame([final_payload["test_metrics"]]).round(4))
        display(pd.DataFrame(final_report).T.round(4))


Output cho thấy điều gì?

Final model được chọn theo validation Macro F1 và chỉ evaluate test một lần. Model `M4 Residual MLP Feature Attention` dùng threshold 0.20; test Accuracy 0.9841, Precision 0.9890, Recall 0.9949, F1 0.9920, Macro F1 0.5295, ROC-AUC 0.6545.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc metric A03 nếu có và so sánh dataset/workflow/metrics với A04. Dataset khác nhau nên không claim score comparison trực tiếp.


Output cho thấy điều gì?

So sánh A03/A04 cho thấy hai notebook giữ cùng target concept `is_churned`, nhưng dữ liệu khác nguồn và khác cách tạo label. Vì vậy score A03 và A04 không phải benchmark trực tiếp; giá trị chính của A04 là dataset thật hơn, workflow DL đầy đủ hơn và kiểm soát rõ hơn các artifact/representation.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        artifact_listing = []


Output cho thấy điều gì?

Notebook đã lưu final model, checkpoint, metrics, figures và metadata dưới `pipeline04/customer_behavior`. Tổng runtime train/evaluate được ghi nhận khoảng 118.23 giây; hạn chế lớn nhất còn lại là Olist có repeat purchase rất thấp, khiến target `is_churned` lệch mạnh và cần được xử lý kỹ ở phần báo cáo.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    from IPython.display import display
    import pandas as pd
    _saved = _repo / 'A04/artifacts/pipeline04/customer_behavior/metrics'
    for _file in sorted(_saved.glob('*comparison*.csv')):
        print(_file.name)
        display(pd.read_csv(_file))


## Thực hành: customer_behavior_torch

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

# A04 customer_behavior — torch

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Customer Behavior: dự đoán customer churn từ hành vi mua hàng và tương tác của khách. A04 dùng Olist Brazilian E-Commerce Public Dataset đã được chuẩn bị thành customer-level churn dataset trong `datasets/customer_behavior/a04`.


Output cho thấy điều gì?

Bảng overview xác nhận A04 giữ đúng business problem của A03: dự đoán `is_churned` cho khách hàng. Dataset A04 là Olist customer-level churn với 58,101 samples, 29 raw input features và target nhị phân; `is_churned=1` nghĩa là khách không có repeat order hợp lệ trong outcome window 180 ngày.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để tránh hard-code đường dẫn cá nhân.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    import os
    import random
    import time
    import warnings
    from datetime import datetime, timezone
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import sklearn
    from IPython.display import display
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    warnings.filterwarnings('ignore', category=FutureWarning)
    SEED = 42
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    ARTIFACT_DIR = PROJECT_ROOT / 'pipeline04' / 'customer_behavior'
    MODEL_DIR = ARTIFACT_DIR / 'models'
    METRIC_DIR = ARTIFACT_DIR / 'metrics'
    FIGURE_DIR = ARTIFACT_DIR / 'figures'
    for directory in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'PyTorch': torch.__version__, 'PyTorch device': str(DEVICE), 'Project root is portable': PROJECT_ROOT.name == 'assign'}
    pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


Output cho thấy điều gì?

Environment có đủ TensorFlow và PyTorch để chạy cùng một prepared customer dataset. CPU vẫn chạy được vì dữ liệu đã aggregate ở cấp customer, không phải event stream thô nhiều triệu dòng.


# 3. Load Dataset

Cell này làm gì?

Liệt kê raw Olist files và đọc prepared customer-level churn CSV. Raw files được giữ nguyên; notebook dùng file prepared để tránh tự tạo target khác A03.


Output cho thấy điều gì?

Folder A04 có raw relational Olist tables và một prepared customer-level file. Notebook đọc `olist_customer_churn_a04.csv` vì file này giữ target `is_churned` đã chốt cho continuity với A03.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra shape, columns, target distribution, missing values, duplicates, descriptive statistics, categorical columns và evidence về sequence/time structure.


Output cho thấy điều gì?

Prepared dataset có 58,101 dòng và 31 cột, không có missing value trong file đã chuẩn bị và không có duplicate row. Target cực kỳ mất cân bằng: class 0 có 665 khách (1.14%), class 1 có 57,436 khách (98.86%); vì vậy notebook báo thêm Macro F1 và Balanced Accuracy để tránh bị accuracy đánh lừa.


# 5. Data Preparation

Cell này làm gì?

Loại identifier, group rare categories theo train only, chia train/validation/test bằng stratify, rồi fit imputer/scaler/one-hot encoder chỉ trên training data.


Output cho thấy điều gì?

Split được stratify thành train/validation/test và preprocessor chỉ fit trên train set. Sau encoding và scaling, mỗi khách hàng được biểu diễn bằng vector tabular gồm 169 features; target test vẫn được giữ riêng cho final evaluation.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của tensor đầu vào deep learning và đánh giá representation có phù hợp với CNN/sequence modeling không.


Output cho thấy điều gì?

Representation chính là customer-level aggregate tabular vector, không phải ảnh và cũng chưa phải event sequence. Raw Olist có timestamped orders/payments/reviews, nhưng notebook này không tự reorder columns để tạo sequence giả; MLP/DNN là baseline tự nhiên, còn Conv1D chỉ được minh họa như operation giáo dục.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa Dense/Linear, ReLU, logits, sigmoid, Conv1D và pooling trên toy tensors.


Output cho thấy điều gì?

Dense tạo logits từ vector customer features, sigmoid chuyển logits thành xác suất churn. Conv1D/pooling chỉ minh họa trên toy sequence có thứ tự rõ, không áp đặt lên các cột aggregate của Olist.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, sigmoid, binary cross entropy, forward composition và Conv1D toy.


Output cho thấy điều gì?

NumPy forward pass minh họa cách MLP tạo xác suất churn từ customer aggregate features. Conv1D thủ công chỉ giải thích operation trên toy sequence, không dùng làm model chính cho prepared Olist table.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric, threshold selection và baseline Keras MLP cho churn classification.


Output cho thấy điều gì?

Keras baseline nhận vector customer behavior đã transform và trả về xác suất `is_churned=1`. Vì label churn là class đa số, notebook chọn threshold theo Macro F1 để buộc model quan tâm cả retained/non-churn minority class.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, class weight, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        def class_weight_from_y(y_array):
            neg = float((y_array == 0).sum())
            pos = float((y_array == 1).sum())
            total = neg + pos
            return {0: total / (2.0 * neg), 1: total / (2.0 * pos)}
        CLASS_WEIGHT = class_weight_from_y(y_train_np)


Output cho thấy điều gì?

Training log cho thấy model học trên train và validation qua từng epoch. Class weight làm class 0 hiếm hơn có ảnh hưởng lớn hơn trong loss, còn test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và AUC của baseline Keras trên train/validation, đồng thời lưu figure.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        def plot_history(history, title, output_path):
            history_df = pd.DataFrame(history.history)
            fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
            axes[0].plot(history_df['loss'], label='train')
            axes[0].plot(history_df['val_loss'], label='validation')
            axes[0].set_title(f'{title} Loss')
            axes[0].set_xlabel('Epoch')
            axes[0].legend()
            axes[1].plot(history_df['auc'], label='train')
            axes[1].plot(history_df['val_auc'], label='validation')
            axes[1].set_title(f'{title} AUC')
            axes[1].set_xlabel('Epoch')
            axes[1].legend()
            fig.tight_layout()
            fig.savefig(output_path, dpi=160)
            plt.show()


Output cho thấy điều gì?

Curves giúp nhìn model có học tín hiệu churn beyond class prior hay không. Với imbalance lớn, validation AUC hữu ích vì không phụ thuộc vào một threshold cố định.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng Accuracy, Precision, Recall, F1, Macro F1, Balanced Accuracy, ROC-AUC, confusion matrix và classification report.


Output cho thấy điều gì?

Keras baseline MLP dùng cùng representation tabular và threshold chọn trên validation. Baseline đạt validation F1 0.9916, Macro F1 0.5281, ROC-AUC 0.5915; positive-class F1 rất cao chủ yếu vì churn là class áp đảo, nên Macro F1 mới là tín hiệu quan trọng hơn.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước train, chạy một batch qua model để kiểm tra input/logits shape.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

        train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train_np.reshape(-1, 1), dtype=torch.float32))
        valid_ds = TensorDataset(torch.tensor(X_valid, dtype=torch.float32), torch.tensor(y_valid_np.reshape(-1, 1), dtype=torch.float32))
        test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test_np.reshape(-1, 1), dtype=torch.float32))

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

        class TorchMLP(nn.Module):
            def __init__(self, input_dim):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(input_dim, 128),
                    nn.ReLU(),
                    nn.Linear(128, 64),
                    nn.ReLU(),
                    nn.Linear(64, 1),
                )

            def forward(self, x):
                return self.net(x)

        def count_torch_params(model):
            return sum(p.numel() for p in model.parameters() if p.requires_grad)

        torch_model = TorchMLP(input_dim).to(DEVICE)
        features_batch, labels_batch = next(iter(train_loader))
        with torch.no_grad():
            logits_batch = torch_model(features_batch.to(DEVICE))

        print("Input feature batch shape:", tuple(features_batch.shape))
        print("Logits batch shape:", tuple(logits_batch.shape))
        print("PyTorch parameter count:", count_torch_params(torch_model))


Output cho thấy điều gì?

Batch input có dạng `(batch, transformed features)` và logits có dạng `(batch, 1)`, đúng với binary churn classification. PyTorch model giữ cùng ý tưởng hidden layers như Keras baseline.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline với loop thủ công, weighted BCE cho imbalance, và lưu checkpoint tốt nhất theo validation ROC-AUC.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        def torch_predict_prob(model, loader):
            model.eval()
            probs = []
            targets = []
            with torch.no_grad():
                for xb, yb in loader:
                    xb = xb.to(DEVICE)
                    logits = model(xb)
                    prob = torch.sigmoid(logits).cpu().numpy().ravel()
                    probs.append(prob)
                    targets.append(yb.numpy().ravel())
            return np.concatenate(targets), np.concatenate(probs)

        neg = float((y_train_np == 0).sum())
        pos = float((y_train_np == 1).sum())
        criterion_base = nn.BCEWithLogitsLoss(reduction="none")
        optimizer = torch.optim.Adam(torch_model.parameters(), lr=1e-3)
        sample_weights = {0: (neg + pos) / (2.0 * neg), 1: (neg + pos) / (2.0 * pos)}

        torch_history_rows = []
        best_auc = -np.inf
        best_state = None
        epochs_without_improvement = 0
        start = time.perf_counter()

        for epoch in range(1, MAX_EPOCHS + 1):
            torch_model.train()
            running_loss = 0.0
            for xb, yb in train_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                optimizer.zero_grad()
                logits = torch_model(xb)
                loss_values = criterion_base(logits, yb)
                weights = torch.where(yb == 1, torch.tensor(sample_weights[1], device=DEVICE), torch.tensor(sample_weights[0], device=DEVICE))
                loss = (loss_values * weights).mean()
                loss.backward()
                optimizer.step()
                running_loss += float(loss.item()) * xb.size(0)

            train_loss = running_loss / len(train_loader.dataset)
            valid_y_epoch, valid_prob_epoch = torch_predict_prob(torch_model, valid_loader)
            valid_auc = roc_auc_score(valid_y_epoch, valid_prob_epoch)
            valid_threshold_epoch, _ = select_threshold(valid_y_epoch, valid_prob_epoch)
            valid_metrics_epoch = evaluate_probabilities(valid_y_epoch, valid_prob_epoch, valid_threshold_epoch)
            torch_history_rows.append(
                {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_auc": valid_auc,
                    "val_macro_f1": valid_metrics_epoch["MacroF1"],
                    "val_balanced_accuracy": valid_metrics_epoch["BalancedAccuracy"],
                }
            )
            print(f"Epoch {epoch:02d} - train_loss={train_loss:.4f} - val_auc={valid_auc:.4f} - val_macro_f1={valid_metrics_epoch['MacroF1']:.4f}")

            if valid_auc > best_auc:
                best_auc = valid_auc
                best_state = {k: v.detach().cpu().clone() for k, v in torch_model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= PATIENCE:
                    print("Early stopping triggered.")
                    break

        torch_train_time = time.perf_counter() - start
        torch_model.load_state_dict(best_state)
        torch_checkpoint = MODEL_DIR / "pytorch_basic_mlp_best.pt"
        torch.save(
            {"model_state_dict": torch_model.state_dict(), "input_dim": input_dim, "feature_columns": feature_cols, "seed": SEED},
            torch_checkpoint,
        )
        torch_history = pd.DataFrame(torch_history_rows)
        torch_history.to_csv(METRIC_DIR / "pytorch_basic_mlp_history.csv", index=False)
        print(f"PyTorch train time: {torch_train_time:.2f} seconds")
        display(torch_history)


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ `zero_grad`, forward, weighted BCE, backward và optimizer step. Vì class 0 rất hiếm, weighted loss giúp model không chỉ tối ưu cho class churn đa số.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng protocol như Keras.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        valid_y_torch, torch_val_prob = torch_predict_prob(torch_model, valid_loader)
        torch_threshold, torch_threshold_table = select_threshold(valid_y_torch, torch_val_prob)
        torch_metrics = evaluate_probabilities(valid_y_torch, torch_val_prob, torch_threshold)
        torch_report = classification_report(
            valid_y_torch.astype(int),
            (torch_val_prob >= torch_threshold).astype(int),
            output_dict=True,
            zero_division=0,
        )

        torch_result = {
            "Model": "M1 Basic MLP",
            "Framework": "PyTorch",
            **torch_metrics,
            "Params": int(count_torch_params(torch_model)),
            "Train Time": float(torch_train_time),
            "Checkpoint": torch_checkpoint.relative_to(PROJECT_ROOT).as_posix(),
        }
        save_json(METRIC_DIR / "pytorch_basic_mlp_validation_metrics.json", torch_result)
        torch_threshold_table.to_csv(METRIC_DIR / "pytorch_basic_mlp_thresholds.csv", index=False)
        pd.DataFrame(torch_report).T.to_csv(METRIC_DIR / "pytorch_basic_mlp_classification_report.csv")

        display(pd.DataFrame([torch_result]).drop(columns=["Checkpoint"]))
        display(pd.DataFrame(confusion_matrix(valid_y_torch, (torch_val_prob >= torch_threshold).astype(int)), index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"]))
        display(pd.DataFrame(torch_report).T.round(4))


Output cho thấy điều gì?

PyTorch được đánh giá trên cùng validation set và cùng threshold rule. Với Olist, retained/non-churn là minority, nên classification report theo từng class cho biết model có bỏ qua class hiếm hay không.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng validation metrics.


Output cho thấy điều gì?

Keras và PyTorch dùng kiến trúc baseline tương đương về ý tưởng nhưng runtime khác nhau trên CPU. PyTorch baseline đạt validation Macro F1 0.5308, Keras baseline đạt 0.5281; đây chỉ là một run thực nghiệm, không phải kết luận framework nào tốt hơn về bản chất.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture MLP phù hợp dữ liệu tabular: M1 Basic MLP, M2 BatchNorm, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được train làm model chính vì input là aggregate tabular.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        def evaluate_keras_model(model, model_key, checkpoint_path, train_time):
            val_prob = model.predict(X_valid, batch_size=BATCH_SIZE, verbose=0).ravel()
            threshold, threshold_table = select_threshold(y_valid_np, val_prob)
            metrics = evaluate_probabilities(y_valid_np, val_prob, threshold)
            result = {'Model': model.name.replace('_', ' '), 'Framework': 'Keras', **metrics, 'Params': int(model.count_params()), 'Train Time': float(train_time), 'Checkpoint': checkpoint_path.relative_to(PROJECT_ROOT).as_posix()}
            save_json(METRIC_DIR / f'{model_key}_validation_metrics.json', result)
            threshold_table.to_csv(METRIC_DIR / f'{model_key}_thresholds.csv', index=False)
            return result


Output cho thấy điều gì?

Các experiment giữ cùng split, preprocessing, threshold rule, seed và training budget. CNN được dừng ở mức investigation vì prepared Olist input không còn event-order structure để Conv1D khai thác.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation metrics và chọn ứng viên tốt nhất bằng validation Macro F1.


Output cho thấy điều gì?

Trong nhóm architecture experiments, model tốt nhất theo validation Macro F1 là `M4 Residual MLP Feature Attention` với Macro F1 0.5293, F1 0.9919, ROC-AUC 0.5907. Chênh lệch giữa các MLP variant nhỏ, nên lợi ích thực tế của kiến trúc phức tạp cần được đọc cùng runtime và imbalance của target.


# 19. Representation Inspection

Cell này làm gì?

Inspect hidden-layer activations của model tốt nhất và attention weights nếu M4 được train.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        hidden_layer = None


Output cho thấy điều gì?

Hidden activations cho biết model tạo latent representation từ behavior aggregates trước output churn. Attention weights, nếu xem từ M4, là tín hiệu model-inspection trên feature đã transform, không phải nguyên nhân kinh doanh chắc chắn.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi validation: confusion matrix, false positives, false negatives và high-confidence errors của model được chọn.


Output cho thấy điều gì?

Error analysis trên validation có 8,575 dự đoán đúng, 95 false positives và 45 false negatives. Với churn, false negative nghĩa là model bỏ sót khách có khả năng churn; tuy nhiên do class churn chiếm gần toàn bộ dataset, cần thận trọng khi diễn giải độ tin cậy của từng nhóm lỗi.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation Macro F1, evaluate test set đúng một lần, rồi lưu final model.


Output cho thấy điều gì?

Final model được chọn theo validation Macro F1 và chỉ evaluate test một lần. Model `M4 Residual MLP Feature Attention` dùng threshold 0.20; test Accuracy 0.9841, Precision 0.9890, Recall 0.9949, F1 0.9920, Macro F1 0.5295, ROC-AUC 0.6545.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc metric A03 nếu có và so sánh dataset/workflow/metrics với A04. Dataset khác nhau nên không claim score comparison trực tiếp.


Output cho thấy điều gì?

So sánh A03/A04 cho thấy hai notebook giữ cùng target concept `is_churned`, nhưng dữ liệu khác nguồn và khác cách tạo label. Vì vậy score A03 và A04 không phải benchmark trực tiếp; giá trị chính của A04 là dataset thật hơn, workflow DL đầy đủ hơn và kiểm soát rõ hơn các artifact/representation.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


Output cho thấy điều gì?

Notebook đã lưu final model, checkpoint, metrics, figures và metadata dưới `pipeline04/customer_behavior`. Tổng runtime train/evaluate được ghi nhận khoảng 118.23 giây; hạn chế lớn nhất còn lại là Olist có repeat purchase rất thấp, khiến target `is_churned` lệch mạnh và cần được xử lý kỹ ở phần báo cáo.


## Thực hành: diabetes_keras

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

# A04 diabetes — keras

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Diabetes: dùng đặc trưng sức khỏe của bệnh nhân/người trả lời khảo sát để dự đoán `Diabetes_binary`. A04 dùng dataset BRFSS 2021 unbalanced đã chuẩn bị trong `datasets/diabetes/a04`, tập trung vào deep learning trên dữ liệu tabular và so sánh cẩn thận với A03.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        from pathlib import Path
        import json
        import pandas as pd


        def find_project_root(start=None):
            start = Path.cwd() if start is None else Path(start)
            for path in [start.resolve(), *start.resolve().parents]:
                if (path / "datasets" / "diabetes" / "a04").exists() and (
                    path / "pipeline"
                ).exists():
                    return path
            raise FileNotFoundError(
                "Cannot locate project root from current working directory."
            )


        PROJECT_ROOT = find_project_root()
        DATA_DIR = PROJECT_ROOT / "datasets" / "diabetes" / "a04"
        A03_NOTEBOOK = PROJECT_ROOT / "pipeline" / "diabetes_pipeline.ipynb"
        A03_METADATA_PATH = PROJECT_ROOT / "pipeline" / "diabetes" / "metadata.json"
        A03_COMPARISON_PATH = PROJECT_ROOT / "pipeline" / "diabetes" / "final_comparison.csv"
        A04_METADATA_PATH = DATA_DIR / "metadata.json"

        with A04_METADATA_PATH.open("r", encoding="utf-8") as f:
            dataset_metadata = json.load(f)

        overview = {
            "A03 task": "Binary diabetes classification from health indicators",
            "A03 target": "Diabetes_binary",
            "A03 notebook exists": A03_NOTEBOOK.exists(),
            "A04 dataset": dataset_metadata["dataset_name"],
            "A04 source": dataset_metadata["source"],
            "A04 target": dataset_metadata["target"],
            "A04 samples": dataset_metadata["samples"],
            "A04 features": dataset_metadata["features"],
            "Task type": "Binary classification",
        }
        pd.DataFrame([overview]).T.rename(columns={0: "Value"})

Output cho thấy điều gì?

Bảng xác nhận notebook giữ đúng continuity với A03: cùng target `Diabetes_binary`, cùng domain patient health indicators, nhưng chuyển sang dataset A04 BRFSS 2021 lớn hơn và mất cân bằng hơn.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để tránh hard-code đường dẫn cá nhân.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    import os
    import random
    import time
    import warnings
    from datetime import datetime, timezone
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import sklearn
    from IPython.display import display
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    warnings.filterwarnings('ignore', category=FutureWarning)
    SEED = 42
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
    tf.get_logger().setLevel('ERROR')
    tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
    ARTIFACT_DIR = PROJECT_ROOT / 'pipeline04' / 'diabetes'
    MODEL_DIR = ARTIFACT_DIR / 'models'
    METRIC_DIR = ARTIFACT_DIR / 'metrics'
    FIGURE_DIR = ARTIFACT_DIR / 'figures'
    for directory in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'TensorFlow': tf.__version__, 'TensorFlow GPUs': len(tf.config.list_physical_devices('GPU')), 'Project root is portable': PROJECT_ROOT.name == 'assign'}
    pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


Output cho thấy điều gì?

Environment có đủ TensorFlow và PyTorch để chạy cùng một pipeline dữ liệu. Nếu device là CPU thì notebook vẫn hợp lệ vì mô hình tabular nhỏ và batch size được chọn để chạy được trên laptop/Colab.


# 3. Load Dataset

Cell này làm gì?

Đọc raw CSV A04 từ `datasets/diabetes/a04` và giữ nguyên file raw. Mọi cleaning/preprocessing chỉ thực hiện trong DataFrame làm việc.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        csv_files = sorted(DATA_DIR.glob("*.csv"))
        raw_files = [
            {
                "file": path.relative_to(PROJECT_ROOT).as_posix(),
                "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
            }
            for path in csv_files
        ]
        display(pd.DataFrame(raw_files))

        DATA_PATH = DATA_DIR / "diabetes_binary_health_indicators_BRFSS2021.csv"
        TARGET = "Diabetes_binary"
        df_raw = pd.read_csv(DATA_PATH)

        print("Loaded dataset:", DATA_PATH.relative_to(PROJECT_ROOT).as_posix())
        print("Shape:", df_raw.shape)
        display(df_raw.head())

Output cho thấy điều gì?

Dataset được đọc từ path tương đối trong project. Shape là số dòng raw và số cột trước khi xử lý duplicate, còn `head()` xác nhận các feature đều là health indicators dạng numeric-coded tabular.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra schema, missing values, duplicates, target distribution, descriptive statistics và các vấn đề dữ liệu rõ ràng của Diabetes.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        shape_info = pd.DataFrame(
            {
                "Metric": ["Rows", "Columns", "Feature columns"],
                "Value": [df_raw.shape[0], df_raw.shape[1], df_raw.shape[1] - 1],
            }
        )

        dtype_info = df_raw.dtypes.astype(str).reset_index()
        dtype_info.columns = ["Column", "Dtype"]

        missing = df_raw.isna().sum().reset_index()
        missing.columns = ["Column", "MissingCount"]
        missing["MissingRate"] = missing["MissingCount"] / len(df_raw)

        duplicate_count = int(df_raw.duplicated().sum())
        target_counts = df_raw[TARGET].value_counts().sort_index()
        target_distribution = pd.DataFrame(
            {
                "Class": target_counts.index.astype(int),
                "Count": target_counts.values,
                "Rate": (target_counts.values / len(df_raw)).round(4),
            }
        )

        clinical_zero_checks = ["BMI", "MentHlth", "PhysHlth"]
        zero_check = pd.DataFrame(
            {
                "Column": clinical_zero_checks,
                "ZeroCount": [int((df_raw[col] == 0).sum()) for col in clinical_zero_checks],
                "Min": [float(df_raw[col].min()) for col in clinical_zero_checks],
                "Max": [float(df_raw[col].max()) for col in clinical_zero_checks],
            }
        )

        display(shape_info)
        display(dtype_info)
        display(
            missing[missing["MissingCount"] > 0]
            if missing["MissingCount"].sum()
            else pd.DataFrame({"Missing": ["No missing values detected"]})
        )
        print("Duplicate rows:", duplicate_count)
        display(target_distribution)
        display(df_raw.describe().T.round(3))
        display(zero_check)

        fig, ax = plt.subplots(figsize=(5.5, 3.5))
        ax.bar(
            target_distribution["Class"].astype(str),
            target_distribution["Count"],
            color=["#4C78A8", "#F58518"],
        )
        ax.set_title("BRFSS 2021 Diabetes_binary Distribution")
        ax.set_xlabel("Diabetes_binary")
        ax.set_ylabel("Count")
        for i, row in target_distribution.iterrows():
            ax.text(i, row["Count"], f'{row["Rate"]:.1%}', ha="center", va="bottom")
        fig.tight_layout()
        target_fig_path = FIGURE_DIR / "target_distribution.png"
        fig.savefig(target_fig_path, dpi=160)
        plt.show()

Output cho thấy điều gì?

Dataset A04 không có missing values nhưng có duplicate rows và class imbalance rõ: class 0 chiếm đa số. BMI không có giá trị 0 vô lý; `MentHlth` và `PhysHlth` có 0 hợp lệ vì đó là số ngày sức khỏe kém trong tháng.


# 5. Data Preparation

Cell này làm gì?

Loại exact duplicates trước split để tránh duplicate leakage, chia train/validation/test theo stratify, rồi fit imputer/scaler chỉ trên training data.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        df_clean = df_raw.drop_duplicates().reset_index(drop=True)
        feature_cols = [col for col in df_clean.columns if col != TARGET]
        numerical_features = feature_cols.copy()
        categorical_features = []

        X = df_clean[feature_cols]
        y = df_clean[TARGET].astype(int)

        X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
            X,
            y,
            test_size=0.30,
            random_state=SEED,
            stratify=y,
        )
        X_valid_raw, X_test_raw, y_valid, y_test = train_test_split(
            X_temp_raw,
            y_temp,
            test_size=0.50,
            random_state=SEED,
            stratify=y_temp,
        )

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="median")),
                            ("scaler", StandardScaler()),
                        ]
                    ),
                    numerical_features,
                )
            ],
            remainder="drop",
        )

        X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
        X_valid = preprocessor.transform(X_valid_raw).astype("float32")
        X_test = preprocessor.transform(X_test_raw).astype("float32")
        y_train_np = y_train.to_numpy(dtype="float32")
        y_valid_np = y_valid.to_numpy(dtype="float32")
        y_test_np = y_test.to_numpy(dtype="float32")

        joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")

        split_summary = pd.DataFrame(
            [
                {
                    "Split": "train",
                    "Rows": len(y_train),
                    "PositiveRate": y_train.mean(),
                    "Negative": int((y_train == 0).sum()),
                    "Positive": int((y_train == 1).sum()),
                },
                {
                    "Split": "validation",
                    "Rows": len(y_valid),
                    "PositiveRate": y_valid.mean(),
                    "Negative": int((y_valid == 0).sum()),
                    "Positive": int((y_valid == 1).sum()),
                },
                {
                    "Split": "test",
                    "Rows": len(y_test),
                    "PositiveRate": y_test.mean(),
                    "Negative": int((y_test == 0).sum()),
                    "Positive": int((y_test == 1).sum()),
                },
            ]
        )

        scale_check = pd.DataFrame(
            {
                "Feature": numerical_features,
                "TrainScaledMean": X_train.mean(axis=0).round(4),
                "TrainScaledStd": X_train.std(axis=0).round(4),
            }
        )

        print("Rows before duplicate removal:", len(df_raw))
        print("Rows after duplicate removal:", len(df_clean))
        print("Removed duplicates:", len(df_raw) - len(df_clean))
        display(split_summary)
        display(scale_check.head(10))

Output cho thấy điều gì?

Split giữ tỷ lệ class gần như giống nhau nhờ stratify. Preprocessor đã `fit` trên train rồi mới transform validation/test, nên không đưa thông tin từ validation/test vào bước scaling.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của tensor đầu vào deep learning và đánh giá representation có phù hợp với CNN không.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        representation_summary = pd.DataFrame(
            {
                "Array": ["X_train", "y_train", "X_valid", "y_valid", "X_test", "y_test"],
                "Shape": [
                    X_train.shape,
                    y_train_np.shape,
                    X_valid.shape,
                    y_valid_np.shape,
                    X_test.shape,
                    y_test_np.shape,
                ],
                "Meaning": [
                    "(samples, features)",
                    "(samples,)",
                    "(samples, features)",
                    "(samples,)",
                    "(samples, features)",
                    "(samples,)",
                ],
            }
        )
        display(representation_summary)

        cnn_suitability = pd.DataFrame(
            [
                {
                    "Question": "Raw representation",
                    "Assessment": "Tabular vector of survey/health indicators",
                },
                {
                    "Question": "Natural baseline",
                    "Assessment": "MLP/DNN, because each sample is a fixed feature vector",
                },
                {
                    "Question": "CNN justified?",
                    "Assessment": "No production CNN: feature columns do not have a stable spatial or local sequential order",
                },
                {
                    "Question": "CNN usage in this notebook",
                    "Assessment": "Only toy Conv1D operations for assignment discovery, not final modeling",
                },
            ]
        )
        display(cnn_suitability)

Output cho thấy điều gì?

`X_train` có dạng `(samples, features)`: mỗi dòng là một người trả lời khảo sát, mỗi cột là một biến sức khỏe. Vì các cột không tạo thành ảnh hay chuỗi có thứ tự lân cận tự nhiên, MLP là baseline đúng phương pháp hơn CNN.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa các operation nhỏ: Dense/Linear, ReLU, logits, sigmoid, Conv1D và pooling trên toy tensors.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        toy_x = tf.constant([[1.0, -2.0, 0.5]], dtype=tf.float32)
        dense = keras.Sequential(
            [
                layers.Input(shape=(3,)),
                layers.Dense(
                    2, use_bias=True, kernel_initializer="ones", bias_initializer="zeros"
                ),
            ]
        )
        toy_logits = dense(toy_x)
        toy_relu = tf.nn.relu(toy_logits)
        toy_sigmoid = tf.nn.sigmoid(toy_logits)

        toy_seq = tf.reshape(tf.range(1, 9, dtype=tf.float32), (1, 8, 1))
        conv = layers.Conv1D(filters=2, kernel_size=3, padding="valid", activation="relu")
        conv_out = conv(toy_seq)
        pool_out = layers.MaxPooling1D(pool_size=2)(conv_out)

        operation_shapes = pd.DataFrame(
            [
                {
                    "Object": "toy_x",
                    "Shape": tuple(toy_x.shape),
                    "Meaning": "one sample, three features",
                },
                {
                    "Object": "Dense logits",
                    "Shape": tuple(toy_logits.shape),
                    "Meaning": "linear outputs before probability",
                },
                {
                    "Object": "ReLU(logits)",
                    "Shape": tuple(toy_relu.shape),
                    "Meaning": "negative values clipped to zero",
                },
                {
                    "Object": "Sigmoid(logits)",
                    "Shape": tuple(toy_sigmoid.shape),
                    "Meaning": "values mapped to 0..1",
                },
                {
                    "Object": "Toy sequence",
                    "Shape": tuple(toy_seq.shape),
                    "Meaning": "(batch, steps, channels)",
                },
                {
                    "Object": "Conv1D output",
                    "Shape": tuple(conv_out.shape),
                    "Meaning": "local windows over toy sequence",
                },
                {
                    "Object": "Pooling output",
                    "Shape": tuple(pool_out.shape),
                    "Meaning": "downsampled Conv1D features",
                },
            ]
        )
        display(operation_shapes)
        print("Dense logits:", toy_logits.numpy().round(4))
        print("Sigmoid probabilities:", toy_sigmoid.numpy().round(4))

Output cho thấy điều gì?

Dense biến vector feature thành logits; sigmoid chuyển logits sang xác suất cho binary classification. Conv1D/pooling chỉ được minh họa trên toy sequence có thứ tự, không áp đặt lên BRFSS tabular features.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, sigmoid, binary cross entropy, forward composition và Conv1D toy.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def linear_np(x, w, b):
            return x @ w + b


        def relu_np(x):
            return np.maximum(0, x)


        def sigmoid_np(x):
            return 1.0 / (1.0 + np.exp(-x))


        def binary_cross_entropy_np(y_true, y_prob, eps=1e-7):
            y_prob = np.clip(y_prob, eps, 1 - eps)
            return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))


        def forward_np(x, params):
            z1 = linear_np(x, params["w1"], params["b1"])
            a1 = relu_np(z1)
            z2 = linear_np(a1, params["w2"], params["b2"])
            return sigmoid_np(z2)


        def conv1d_valid_np(sequence, kernel, bias=0.0):
            sequence = np.asarray(sequence, dtype=float)
            kernel = np.asarray(kernel, dtype=float)
            out = []
            for i in range(len(sequence) - len(kernel) + 1):
                out.append(float(np.sum(sequence[i : i + len(kernel)] * kernel) + bias))
            return np.asarray(out)


        rng = np.random.default_rng(SEED)
        toy_batch = X_train[:4]
        params = {
            "w1": rng.normal(0, 0.1, size=(X_train.shape[1], 5)),
            "b1": np.zeros(5),
            "w2": rng.normal(0, 0.1, size=(5, 1)),
            "b2": np.zeros(1),
        }
        toy_probs = forward_np(toy_batch, params).ravel()
        toy_loss = binary_cross_entropy_np(y_train_np[:4], toy_probs)
        toy_conv = conv1d_valid_np([1, 2, 3, 4, 5], [0.25, 0.5, 0.25])

        print("Forward probabilities:", np.round(toy_probs, 4))
        print("Toy BCE:", round(float(toy_loss), 4))
        print("Manual Conv1D toy output:", np.round(toy_conv, 4))

Output cho thấy điều gì?

NumPy forward pass tạo xác suất từ tabular features giống ý tưởng DNN trong A03. Conv1D thủ công chỉ minh họa phép trượt kernel trên một vector toy có thứ tự, không dùng như mô hình chính cho dataset BRFSS.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric, threshold selection và baseline Keras MLP cho Diabetes.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def to_jsonable(obj):
            if isinstance(obj, dict):
                return {str(k): to_jsonable(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [to_jsonable(v) for v in obj]
            if isinstance(obj, tuple):
                return [to_jsonable(v) for v in obj]
            if isinstance(obj, (np.integer,)):
                return int(obj)
            if isinstance(obj, (np.floating,)):
                return float(obj)
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            return obj


        def select_threshold(y_true, prob, metric="f1"):
            thresholds = np.round(np.linspace(0.10, 0.90, 81), 2)
            rows = []
            for threshold in thresholds:
                pred = (prob >= threshold).astype(int)
                rows.append(
                    {
                        "Threshold": float(threshold),
                        "Accuracy": accuracy_score(y_true, pred),
                        "Precision": precision_score(y_true, pred, zero_division=0),
                        "Recall": recall_score(y_true, pred, zero_division=0),
                        "F1": f1_score(y_true, pred, zero_division=0),
                    }
                )
            table = pd.DataFrame(rows)
            best = table.sort_values(["F1", "Recall", "Precision"], ascending=False).iloc[0]
            return float(best["Threshold"]), table


        def evaluate_probabilities(y_true, prob, threshold=None):
            if threshold is None:
                threshold, _ = select_threshold(y_true, prob)
            pred = (prob >= threshold).astype(int)
            cm = confusion_matrix(y_true, pred)
            tn, fp, fn, tp = cm.ravel()
            return {
                "Threshold": float(threshold),
                "Accuracy": float(accuracy_score(y_true, pred)),
                "Precision": float(precision_score(y_true, pred, zero_division=0)),
                "Recall": float(recall_score(y_true, pred, zero_division=0)),
                "F1": float(f1_score(y_true, pred, zero_division=0)),
                "ROC-AUC": float(roc_auc_score(y_true, prob)),
                "TN": int(tn),
                "FP": int(fp),
                "FN": int(fn),
                "TP": int(tp),
            }


        def save_json(path, obj):
            with Path(path).open("w", encoding="utf-8") as f:
                json.dump(to_jsonable(obj), f, indent=2)


        def build_basic_mlp(input_dim, name="M1_Basic_MLP"):
            inputs = keras.Input(shape=(input_dim,), name="health_indicator_vector")
            x = layers.Dense(64, activation="relu", name="dense_64")(inputs)
            x = layers.Dense(32, activation="relu", name="dense_32")(x)
            outputs = layers.Dense(1, activation="sigmoid", name="diabetes_probability")(x)
            model = keras.Model(inputs, outputs, name=name)
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                loss="binary_crossentropy",
                metrics=[
                    keras.metrics.BinaryAccuracy(name="accuracy"),
                    keras.metrics.AUC(name="auc"),
                ],
            )
            return model


        input_dim = X_train.shape[1]
        keras_baseline = build_basic_mlp(input_dim)
        keras_baseline.summary()
        print("Total parameters:", keras_baseline.count_params())

Output cho thấy điều gì?

Keras baseline nhận input 21 feature và trả về một xác suất `Diabetes_binary=1`. Số parameter nhỏ so với số sample, phù hợp với dữ liệu tabular và giảm nguy cơ overfitting quá mạnh.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, class weight, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def class_weight_from_y(y_array):
            neg = float((y_array == 0).sum())
            pos = float((y_array == 1).sum())
            total = neg + pos
            return {0: total / (2.0 * neg), 1: total / (2.0 * pos)}


        CLASS_WEIGHT = class_weight_from_y(y_train_np)
        BATCH_SIZE = 4096
        MAX_EPOCHS = 12
        PATIENCE = 2


        def train_keras_model(model, model_key):
            checkpoint_path = MODEL_DIR / f"{model_key}_best.keras"
            callbacks = [
                keras.callbacks.EarlyStopping(
                    monitor="val_auc", mode="max", patience=PATIENCE, restore_best_weights=True
                ),
                keras.callbacks.ModelCheckpoint(
                    checkpoint_path, monitor="val_auc", mode="max", save_best_only=True
                ),
            ]
            start = time.perf_counter()
            history = model.fit(
                X_train,
                y_train_np,
                validation_data=(X_valid, y_valid_np),
                epochs=MAX_EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=callbacks,
                class_weight=CLASS_WEIGHT,
                verbose=2,
            )
            train_time = time.perf_counter() - start
            history_df = pd.DataFrame(history.history)
            history_df.to_csv(METRIC_DIR / f"{model_key}_history.csv", index=False)
            return history, train_time, checkpoint_path


        keras_history, keras_train_time, keras_checkpoint = train_keras_model(
            keras_baseline, "m1_basic_mlp"
        )
        print(f"Keras baseline train time: {keras_train_time:.2f} seconds")
        print("Best checkpoint:", keras_checkpoint.relative_to(PROJECT_ROOT).as_posix())
        display(pd.DataFrame(keras_history.history).tail())

Output cho thấy điều gì?

Training log cho thấy loss/metric theo từng epoch trên train và validation. EarlyStopping/ModelCheckpoint chọn epoch có validation AUC tốt nhất, còn test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và AUC của baseline Keras trên train/validation, đồng thời lưu figure.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def plot_history(history, title, output_path):
            history_df = pd.DataFrame(history.history)
            fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
            axes[0].plot(history_df["loss"], label="train")
            axes[0].plot(history_df["val_loss"], label="validation")
            axes[0].set_title(f"{title} Loss")
            axes[0].set_xlabel("Epoch")
            axes[0].legend()
            axes[1].plot(history_df["auc"], label="train")
            axes[1].plot(history_df["val_auc"], label="validation")
            axes[1].set_title(f"{title} AUC")
            axes[1].set_xlabel("Epoch")
            axes[1].legend()
            fig.tight_layout()
            fig.savefig(output_path, dpi=160)
            plt.show()


        plot_history(
            keras_history, "Keras Basic MLP", FIGURE_DIR / "keras_basic_mlp_training_curves.png"
        )

Output cho thấy điều gì?

Curves giúp kiểm tra mô hình học ổn định hay overfit. Khoảng cách train/validation là tín hiệu quan trọng hơn một epoch đơn lẻ.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng threshold chọn từ validation F1, gồm Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix và classification report.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        keras_val_prob = keras_baseline.predict(
            X_valid, batch_size=BATCH_SIZE, verbose=0
        ).ravel()
        keras_threshold, keras_threshold_table = select_threshold(y_valid_np, keras_val_prob)
        keras_metrics = evaluate_probabilities(y_valid_np, keras_val_prob, keras_threshold)
        keras_report = classification_report(
            y_valid_np.astype(int),
            (keras_val_prob >= keras_threshold).astype(int),
            output_dict=True,
            zero_division=0,
        )

        keras_result = {
            "Model": "M1 Basic MLP",
            "Framework": "Keras",
            **keras_metrics,
            "Params": int(keras_baseline.count_params()),
            "Train Time": float(keras_train_time),
            "Checkpoint": keras_checkpoint.relative_to(PROJECT_ROOT).as_posix(),
        }

        save_json(METRIC_DIR / "keras_basic_mlp_validation_metrics.json", keras_result)
        keras_threshold_table.to_csv(METRIC_DIR / "keras_basic_mlp_thresholds.csv", index=False)
        pd.DataFrame(keras_report).T.to_csv(
            METRIC_DIR / "keras_basic_mlp_classification_report.csv"
        )

        display(pd.DataFrame([keras_result]).drop(columns=["Checkpoint"]))
        display(
            pd.DataFrame(
                confusion_matrix(y_valid_np, (keras_val_prob >= keras_threshold).astype(int)),
                index=["Actual 0", "Actual 1"],
                columns=["Pred 0", "Pred 1"],
            )
        )
        display(pd.DataFrame(keras_report).T.round(4))

Output cho thấy điều gì?

Validation metrics dùng nhiều góc nhìn, không chỉ accuracy. Với bài toán diabetes, false negatives đáng chú ý vì là các ca có diabetes/risk nhưng model dự đoán âm tính; tuy vậy đây vẫn là phân tích trên survey dataset, không phải chẩn đoán y khoa.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước khi train, chạy một batch qua model để kiểm tra input/logits shape.


Output cho thấy điều gì?

Batch input có dạng `(batch, 21)` và logits có dạng `(batch, 1)`, đúng với binary classification. PyTorch model dùng architecture tương đương Keras baseline để so sánh framework công bằng hơn.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline với loop thủ công, class imbalance dùng `pos_weight`, và lưu checkpoint tốt nhất theo validation ROC-AUC.


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ các bước `zero_grad`, forward, loss, backward và optimizer step. Checkpoint tốt nhất được chọn bằng validation ROC-AUC, không dùng test set.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng protocol như Keras.


Output cho thấy điều gì?

PyTorch được đánh giá trên cùng validation set và cùng cách chọn threshold theo F1. Vì framework, initialization và optimizer implementation có khác biệt nhỏ, một run không đủ để kết luận framework nào tốt hơn về bản chất.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng cùng metric validation.


Output cho thấy điều gì?

Bảng chỉ so sánh hai implementation baseline trong điều kiện hiện tại. Chênh lệch nhỏ có thể đến từ initialization, training loop và numerical behavior; không nên kết luận framework nào intrinsically better từ một run.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture MLP phù hợp dữ liệu tabular: M1 Basic MLP, M2 BatchNorm MLP, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được ép vào dữ liệu BRFSS vì feature order không có local structure tự nhiên.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def compile_binary_model(model):
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                loss="binary_crossentropy",
                metrics=[
                    keras.metrics.BinaryAccuracy(name="accuracy"),
                    keras.metrics.AUC(name="auc"),
                ],
            )
            return model


        def build_batchnorm_mlp(input_dim, name="M2_BatchNorm_MLP"):
            inputs = keras.Input(shape=(input_dim,), name="health_indicator_vector")
            x = layers.Dense(64, use_bias=False, name="dense_64")(inputs)
            x = layers.BatchNormalization(name="bn_64")(x)
            x = layers.Activation("relu", name="relu_64")(x)
            x = layers.Dense(32, use_bias=False, name="dense_32")(x)
            x = layers.BatchNormalization(name="bn_32")(x)
            x = layers.Activation("relu", name="relu_32")(x)
            outputs = layers.Dense(1, activation="sigmoid", name="diabetes_probability")(x)
            return compile_binary_model(keras.Model(inputs, outputs, name=name))


        def residual_block(x, units, block_name):
            shortcut = x
            y = layers.Dense(units, activation="relu", name=f"{block_name}_dense_a")(x)
            y = layers.Dense(units, name=f"{block_name}_dense_b")(y)
            x = layers.Add(name=f"{block_name}_add")([shortcut, y])
            return layers.Activation("relu", name=f"{block_name}_relu")(x)


        def build_residual_mlp(input_dim, name="M3_Residual_MLP"):
            inputs = keras.Input(shape=(input_dim,), name="health_indicator_vector")
            x = layers.Dense(64, activation="relu", name="input_projection")(inputs)
            x = residual_block(x, 64, "residual_1")
            x = residual_block(x, 64, "residual_2")
            x = layers.Dense(32, activation="relu", name="dense_32")(x)
            outputs = layers.Dense(1, activation="sigmoid", name="diabetes_probability")(x)
            return compile_binary_model(keras.Model(inputs, outputs, name=name))


        def build_attention_residual_mlp(input_dim, name="M4_Residual_MLP_Feature_Attention"):
            inputs = keras.Input(shape=(input_dim,), name="health_indicator_vector")
            attention = layers.Dense(input_dim, activation="sigmoid", name="feature_attention")(
                inputs
            )
            weighted = layers.Multiply(name="attention_weighted_features")([inputs, attention])
            x = layers.Dense(64, activation="relu", name="input_projection")(weighted)
            x = residual_block(x, 64, "residual_1")
            x = residual_block(x, 64, "residual_2")
            x = layers.Dense(32, activation="relu", name="dense_32")(x)
            outputs = layers.Dense(1, activation="sigmoid", name="diabetes_probability")(x)
            return compile_binary_model(keras.Model(inputs, outputs, name=name))


        def evaluate_keras_model(model, model_key, checkpoint_path, train_time):
            val_prob = model.predict(X_valid, batch_size=BATCH_SIZE, verbose=0).ravel()
            threshold, threshold_table = select_threshold(y_valid_np, val_prob)
            metrics = evaluate_probabilities(y_valid_np, val_prob, threshold)
            result = {
                "Model": model.name.replace("_", " "),
                "Framework": "Keras",
                **metrics,
                "Params": int(model.count_params()),
                "Train Time": float(train_time),
                "Checkpoint": checkpoint_path.relative_to(PROJECT_ROOT).as_posix(),
            }
            save_json(METRIC_DIR / f"{model_key}_validation_metrics.json", result)
            threshold_table.to_csv(METRIC_DIR / f"{model_key}_thresholds.csv", index=False)
            return result


        architecture_results = [keras_result]
        architecture_histories = {"M1 Basic MLP": keras_history}
        trained_keras_models = {"M1 Basic MLP": keras_baseline}

        for model_key, builder in [
            ("m2_batchnorm_mlp", build_batchnorm_mlp),
            ("m3_residual_mlp", build_residual_mlp),
            ("m4_attention_residual_mlp", build_attention_residual_mlp),
        ]:
            keras.backend.clear_session()
            tf.random.set_seed(SEED)
            model = builder(input_dim)
            print("\nTraining", model.name, "with", model.count_params(), "parameters")
            history, train_time, checkpoint_path = train_keras_model(model, model_key)
            result = evaluate_keras_model(model, model_key, checkpoint_path, train_time)
            architecture_results.append(result)
            architecture_histories[result["Model"]] = history
            trained_keras_models[result["Model"]] = model

        cnn_investigation = pd.DataFrame(
            [
                {
                    "Investigation": "CNN on BRFSS tabular features",
                    "Decision": "Not trained as a candidate model",
                    "Reason": "Feature columns do not have meaningful local ordering; Conv1D/Conv2D would create artificial structure.",
                }
            ]
        )
        display(cnn_investigation)

Output cho thấy điều gì?

Các experiment giữ cùng split, preprocessing, seed và training budget. CNN được xử lý như phần discovery/toy operation, không đưa vào model chính vì không có cơ sở spatial/sequential trên raw representation.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation metrics và chọn ứng viên tốt nhất bằng validation F1.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        architecture_comparison = pd.DataFrame(architecture_results)[
            [
                "Model",
                "Accuracy",
                "Precision",
                "Recall",
                "F1",
                "ROC-AUC",
                "Params",
                "Train Time",
                "Threshold",
                "Checkpoint",
            ]
        ].sort_values("F1", ascending=False)
        architecture_comparison.to_csv(METRIC_DIR / "architecture_comparison.csv", index=False)

        display(architecture_comparison.drop(columns=["Checkpoint"]).round(4))

        fig, ax = plt.subplots(figsize=(8, 3.8))
        plot_df = architecture_comparison.sort_values("F1")
        ax.barh(plot_df["Model"], plot_df["F1"], color="#4C78A8")
        ax.set_xlabel("Validation F1")
        ax.set_title("A04 Diabetes Architecture Comparison")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "architecture_comparison_f1.png", dpi=160)
        plt.show()

        best_validation_row = architecture_comparison.iloc[0].to_dict()
        print("Best validation model:", best_validation_row["Model"])
        print("Best validation F1:", round(best_validation_row["F1"], 4))

Output cho thấy điều gì?

Bảng xếp hạng theo validation F1 vì dataset mất cân bằng và chỉ accuracy sẽ thiên về class 0. Trade-off cần đọc cùng Precision/Recall: recall cao hơn thường giảm false negatives nhưng có thể tăng false positives.


# 19. Representation Inspection

Cell này làm gì?

Inspect representation bên trong: activation của hidden layer cho model tốt nhất và attention weights nếu M4 được train.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        best_model_name = best_validation_row["Model"]
        best_checkpoint_path = PROJECT_ROOT / best_validation_row["Checkpoint"]
        best_validation_model = keras.models.load_model(best_checkpoint_path)

        hidden_layer = None
        for layer in best_validation_model.layers:
            if isinstance(layer, layers.Dense) and getattr(layer, "units", None) != 1:
                hidden_layer = layer.name
                break

        if hidden_layer is not None:
            activation_model = keras.Model(
                best_validation_model.input,
                best_validation_model.get_layer(hidden_layer).output,
            )
            sample_activations = activation_model.predict(
                X_valid[:1024], batch_size=1024, verbose=0
            )
            activation_summary = pd.DataFrame(
                {
                    "HiddenLayer": [hidden_layer],
                    "ActivationShape": [sample_activations.shape],
                    "MeanActivation": [float(sample_activations.mean())],
                    "StdActivation": [float(sample_activations.std())],
                    "ZeroRate": [float((np.abs(sample_activations) < 1e-8).mean())],
                }
            )
            display(activation_summary.round(4))

        attention_model_path = MODEL_DIR / "m4_attention_residual_mlp_best.keras"
        if attention_model_path.exists():
            attention_model = keras.models.load_model(attention_model_path)
            attention_extractor = keras.Model(
                attention_model.input, attention_model.get_layer("feature_attention").output
            )
            attention_values = attention_extractor.predict(
                X_valid[:5000], batch_size=4096, verbose=0
            )
            attention_summary = pd.DataFrame(
                {
                    "Feature": feature_cols,
                    "MeanAttention": attention_values.mean(axis=0),
                    "StdAttention": attention_values.std(axis=0),
                }
            ).sort_values("MeanAttention", ascending=False)
            attention_summary.to_csv(METRIC_DIR / "feature_attention_summary.csv", index=False)
            display(attention_summary.head(10).round(4))
            fig, ax = plt.subplots(figsize=(8, 5))
            top_attention = attention_summary.head(12).sort_values("MeanAttention")
            ax.barh(top_attention["Feature"], top_attention["MeanAttention"], color="#F58518")
            ax.set_title("Top Feature Attention Weights")
            ax.set_xlabel("Mean attention")
            fig.tight_layout()
            fig.savefig(FIGURE_DIR / "feature_attention_weights.png", dpi=160)
            plt.show()

Output cho thấy điều gì?

Hidden activations cho biết tabular vector được model biến đổi thành representation dày hơn trước output. Nếu có attention summary, đó là inspection phụ trợ về trọng số tương đối của feature trong M4, không phải bằng chứng nhân quả y khoa.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi trên validation set: false positives, false negatives và high-confidence errors của model được chọn theo validation.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        best_val_prob = best_validation_model.predict(
            X_valid, batch_size=BATCH_SIZE, verbose=0
        ).ravel()
        best_threshold = float(best_validation_row["Threshold"])
        best_val_pred = (best_val_prob >= best_threshold).astype(int)

        error_df = X_valid_raw.reset_index(drop=True).copy()
        error_df["y_true"] = y_valid_np.astype(int)
        error_df["prob_diabetes"] = best_val_prob
        error_df["y_pred"] = best_val_pred
        error_df["confidence"] = np.where(best_val_pred == 1, best_val_prob, 1 - best_val_prob)
        error_df["error_type"] = np.select(
            [
                (error_df["y_true"] == 0) & (error_df["y_pred"] == 1),
                (error_df["y_true"] == 1) & (error_df["y_pred"] == 0),
            ],
            ["False Positive", "False Negative"],
            default="Correct",
        )

        error_summary = (
            error_df["error_type"]
            .value_counts()
            .rename_axis("ErrorType")
            .reset_index(name="Count")
        )
        high_conf_errors = (
            error_df[error_df["error_type"] != "Correct"]
            .sort_values("confidence", ascending=False)
            .head(10)
        )
        fp_fn_profile = (
            error_df[error_df["error_type"].isin(["False Positive", "False Negative"])]
            .groupby("error_type")[feature_cols]
            .mean()
            .T
        )

        error_summary.to_csv(METRIC_DIR / "validation_error_summary.csv", index=False)
        high_conf_errors.to_csv(
            METRIC_DIR / "validation_high_confidence_errors.csv", index=False
        )
        fp_fn_profile.to_csv(METRIC_DIR / "validation_fp_fn_feature_means.csv")

        display(error_summary)
        display(
            high_conf_errors[[TARGET for TARGET in []]]
            if False
            else high_conf_errors[
                [
                    "y_true",
                    "y_pred",
                    "prob_diabetes",
                    "confidence",
                    "error_type",
                    "HighBP",
                    "HighChol",
                    "BMI",
                    "GenHlth",
                    "Age",
                ]
            ]
        )
        display(
            fp_fn_profile.loc[
                ["HighBP", "HighChol", "BMI", "GenHlth", "DiffWalk", "Age", "Income"]
            ].round(3)
        )

Output cho thấy điều gì?

False negatives là nhóm cần chú ý trong bài toán diabetes vì model bỏ sót class 1 trên validation. High-confidence errors giúp nhận diện các vùng dữ liệu model tự tin nhưng sai; các nhận xét này chỉ dùng cho phân tích mô hình, không phải khuyến nghị lâm sàng.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation, evaluate test set đúng một lần, rồi lưu final model và metrics.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        final_model = keras.models.load_model(best_checkpoint_path)
        test_prob = final_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
        final_threshold = float(best_validation_row["Threshold"])
        final_metrics = evaluate_probabilities(y_test_np, test_prob, final_threshold)
        final_pred = (test_prob >= final_threshold).astype(int)
        final_report = classification_report(
            y_test_np.astype(int), final_pred, output_dict=True, zero_division=0
        )

        final_model_path = MODEL_DIR / "final_model.keras"
        final_model.save(final_model_path)

        final_payload = {
            "selected_model": best_validation_row["Model"],
            "selection_metric": "Validation F1",
            "decision_threshold": final_threshold,
            "test_metrics": final_metrics,
            "final_model_path": final_model_path.relative_to(PROJECT_ROOT).as_posix(),
            "source_checkpoint": best_checkpoint_path.relative_to(PROJECT_ROOT).as_posix(),
        }
        save_json(METRIC_DIR / "final_test_metrics.json", final_payload)
        pd.DataFrame(final_report).T.to_csv(METRIC_DIR / "final_classification_report.csv")

        final_cm = confusion_matrix(y_test_np, final_pred)
        fig, ax = plt.subplots(figsize=(4.5, 4))
        im = ax.imshow(final_cm, cmap="Blues")
        ax.set_xticks([0, 1], labels=["Pred 0", "Pred 1"])
        ax.set_yticks([0, 1], labels=["Actual 0", "Actual 1"])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(final_cm[i, j]), ha="center", va="center", color="black")
        ax.set_title("Final Test Confusion Matrix")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "final_confusion_matrix.png", dpi=160)
        plt.show()

        fpr, tpr, _ = roc_curve(y_test_np, test_prob)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(fpr, tpr, label=f"ROC-AUC={final_metrics['ROC-AUC']:.4f}")
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("Final Test ROC Curve")
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "final_roc_curve.png", dpi=160)
        plt.show()

        display(pd.DataFrame([final_payload["test_metrics"]]).round(4))
        display(pd.DataFrame(final_report).T.round(4))

Output cho thấy điều gì?

Đây là kết quả test cuối cùng của model đã chọn bằng validation. `M3 Residual MLP` dùng threshold 0.63; test Accuracy 0.7844, Precision 0.3670, Recall 0.6084, F1 0.4578, ROC-AUC 0.8089.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc artifacts A03 nếu có và so sánh dataset/workflow/metrics với A04. Không kết luận A04 tốt hơn chỉ vì score khác, vì dataset đã đổi.


Output cho thấy điều gì?

A03 và A04 cùng bài toán nhưng khác dataset: A03 cân bằng hơn, A04 thực tế hơn và mất cân bằng hơn. Vì vậy metric không chứng minh deep learning intrinsically better/worse; nó chỉ mô tả workflow và kết quả trên dữ liệu A04 đã chọn.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


Output cho thấy điều gì?

Bảng cuối ghi lại dataset, model được chọn, threshold và test metrics thật. Artifact listing xác nhận notebook đã lưu model, metrics, figures và metadata cho phase Diabetes A04; recorded training/evaluation time khoảng 306.08 giây.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    from IPython.display import display
    import pandas as pd
    _saved = _repo / 'A04/artifacts/pipeline04/diabetes/metrics'
    for _file in sorted(_saved.glob('*comparison*.csv')):
        print(_file.name)
        display(pd.read_csv(_file))


## Thực hành: diabetes_torch

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

# A04 diabetes — torch

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Diabetes: dùng đặc trưng sức khỏe của bệnh nhân/người trả lời khảo sát để dự đoán `Diabetes_binary`. A04 dùng dataset BRFSS 2021 unbalanced đã chuẩn bị trong `datasets/diabetes/a04`, tập trung vào deep learning trên dữ liệu tabular và so sánh cẩn thận với A03.


Output cho thấy điều gì?

Bảng xác nhận notebook giữ đúng continuity với A03: cùng target `Diabetes_binary`, cùng domain patient health indicators, nhưng chuyển sang dataset A04 BRFSS 2021 lớn hơn và mất cân bằng hơn.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để tránh hard-code đường dẫn cá nhân.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    import os
    import random
    import time
    import warnings
    from datetime import datetime, timezone
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import sklearn
    from IPython.display import display
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    warnings.filterwarnings('ignore', category=FutureWarning)
    SEED = 42
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    ARTIFACT_DIR = PROJECT_ROOT / 'pipeline04' / 'diabetes'
    MODEL_DIR = ARTIFACT_DIR / 'models'
    METRIC_DIR = ARTIFACT_DIR / 'metrics'
    FIGURE_DIR = ARTIFACT_DIR / 'figures'
    for directory in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'PyTorch': torch.__version__, 'PyTorch device': str(DEVICE), 'Project root is portable': PROJECT_ROOT.name == 'assign'}
    pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


Output cho thấy điều gì?

Environment có đủ TensorFlow và PyTorch để chạy cùng một pipeline dữ liệu. Nếu device là CPU thì notebook vẫn hợp lệ vì mô hình tabular nhỏ và batch size được chọn để chạy được trên laptop/Colab.


# 3. Load Dataset

Cell này làm gì?

Đọc raw CSV A04 từ `datasets/diabetes/a04` và giữ nguyên file raw. Mọi cleaning/preprocessing chỉ thực hiện trong DataFrame làm việc.


Output cho thấy điều gì?

Dataset được đọc từ path tương đối trong project. Shape là số dòng raw và số cột trước khi xử lý duplicate, còn `head()` xác nhận các feature đều là health indicators dạng numeric-coded tabular.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra schema, missing values, duplicates, target distribution, descriptive statistics và các vấn đề dữ liệu rõ ràng của Diabetes.


Output cho thấy điều gì?

Dataset A04 không có missing values nhưng có duplicate rows và class imbalance rõ: class 0 chiếm đa số. BMI không có giá trị 0 vô lý; `MentHlth` và `PhysHlth` có 0 hợp lệ vì đó là số ngày sức khỏe kém trong tháng.


# 5. Data Preparation

Cell này làm gì?

Loại exact duplicates trước split để tránh duplicate leakage, chia train/validation/test theo stratify, rồi fit imputer/scaler chỉ trên training data.


Output cho thấy điều gì?

Split giữ tỷ lệ class gần như giống nhau nhờ stratify. Preprocessor đã `fit` trên train rồi mới transform validation/test, nên không đưa thông tin từ validation/test vào bước scaling.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của tensor đầu vào deep learning và đánh giá representation có phù hợp với CNN không.


Output cho thấy điều gì?

`X_train` có dạng `(samples, features)`: mỗi dòng là một người trả lời khảo sát, mỗi cột là một biến sức khỏe. Vì các cột không tạo thành ảnh hay chuỗi có thứ tự lân cận tự nhiên, MLP là baseline đúng phương pháp hơn CNN.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa các operation nhỏ: Dense/Linear, ReLU, logits, sigmoid, Conv1D và pooling trên toy tensors.


Output cho thấy điều gì?

Dense biến vector feature thành logits; sigmoid chuyển logits sang xác suất cho binary classification. Conv1D/pooling chỉ được minh họa trên toy sequence có thứ tự, không áp đặt lên BRFSS tabular features.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, sigmoid, binary cross entropy, forward composition và Conv1D toy.


Output cho thấy điều gì?

NumPy forward pass tạo xác suất từ tabular features giống ý tưởng DNN trong A03. Conv1D thủ công chỉ minh họa phép trượt kernel trên một vector toy có thứ tự, không dùng như mô hình chính cho dataset BRFSS.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric, threshold selection và baseline Keras MLP cho Diabetes.


Output cho thấy điều gì?

Keras baseline nhận input 21 feature và trả về một xác suất `Diabetes_binary=1`. Số parameter nhỏ so với số sample, phù hợp với dữ liệu tabular và giảm nguy cơ overfitting quá mạnh.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, class weight, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        def class_weight_from_y(y_array):
            neg = float((y_array == 0).sum())
            pos = float((y_array == 1).sum())
            total = neg + pos
            return {0: total / (2.0 * neg), 1: total / (2.0 * pos)}
        CLASS_WEIGHT = class_weight_from_y(y_train_np)
        BATCH_SIZE = 4096
        MAX_EPOCHS = 12
        PATIENCE = 2


Output cho thấy điều gì?

Training log cho thấy loss/metric theo từng epoch trên train và validation. EarlyStopping/ModelCheckpoint chọn epoch có validation AUC tốt nhất, còn test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và AUC của baseline Keras trên train/validation, đồng thời lưu figure.


Output cho thấy điều gì?

Curves giúp kiểm tra mô hình học ổn định hay overfit. Khoảng cách train/validation là tín hiệu quan trọng hơn một epoch đơn lẻ.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng threshold chọn từ validation F1, gồm Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix và classification report.


Output cho thấy điều gì?

Validation metrics dùng nhiều góc nhìn, không chỉ accuracy. Với bài toán diabetes, false negatives đáng chú ý vì là các ca có diabetes/risk nhưng model dự đoán âm tính; tuy vậy đây vẫn là phân tích trên survey dataset, không phải chẩn đoán y khoa.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước khi train, chạy một batch qua model để kiểm tra input/logits shape.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

        train_ds = TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train_np.reshape(-1, 1), dtype=torch.float32),
        )
        valid_ds = TensorDataset(
            torch.tensor(X_valid, dtype=torch.float32),
            torch.tensor(y_valid_np.reshape(-1, 1), dtype=torch.float32),
        )
        test_ds = TensorDataset(
            torch.tensor(X_test, dtype=torch.float32),
            torch.tensor(y_test_np.reshape(-1, 1), dtype=torch.float32),
        )

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


        class TorchMLP(nn.Module):
            def __init__(self, input_dim):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(input_dim, 64),
                    nn.ReLU(),
                    nn.Linear(64, 32),
                    nn.ReLU(),
                    nn.Linear(32, 1),
                )

            def forward(self, x):
                return self.net(x)


        def count_torch_params(model):
            return sum(p.numel() for p in model.parameters() if p.requires_grad)


        torch_model = TorchMLP(input_dim).to(DEVICE)
        features_batch, labels_batch = next(iter(train_loader))
        with torch.no_grad():
            logits_batch = torch_model(features_batch.to(DEVICE))

        print("Input feature batch shape:", tuple(features_batch.shape))
        print("Logits batch shape:", tuple(logits_batch.shape))
        print("PyTorch parameter count:", count_torch_params(torch_model))

Output cho thấy điều gì?

Batch input có dạng `(batch, 21)` và logits có dạng `(batch, 1)`, đúng với binary classification. PyTorch model dùng architecture tương đương Keras baseline để so sánh framework công bằng hơn.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline với loop thủ công, class imbalance dùng `pos_weight`, và lưu checkpoint tốt nhất theo validation ROC-AUC.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        def torch_predict_prob(model, loader):
            model.eval()
            probs = []
            targets = []
            with torch.no_grad():
                for xb, yb in loader:
                    xb = xb.to(DEVICE)
                    logits = model(xb)
                    prob = torch.sigmoid(logits).cpu().numpy().ravel()
                    probs.append(prob)
                    targets.append(yb.numpy().ravel())
            return np.concatenate(targets), np.concatenate(probs)


        neg_count = float((y_train_np == 0).sum())
        pos_count = float((y_train_np == 1).sum())
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([neg_count / pos_count], dtype=torch.float32, device=DEVICE)
        )
        optimizer = torch.optim.Adam(torch_model.parameters(), lr=1e-3)

        torch_history_rows = []
        best_auc = -np.inf
        best_state = None
        epochs_without_improvement = 0
        start = time.perf_counter()

        for epoch in range(1, MAX_EPOCHS + 1):
            torch_model.train()
            running_loss = 0.0
            for xb, yb in train_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                optimizer.zero_grad()
                logits = torch_model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()
                running_loss += float(loss.item()) * xb.size(0)

            train_loss = running_loss / len(train_loader.dataset)
            valid_y_epoch, valid_prob_epoch = torch_predict_prob(torch_model, valid_loader)
            valid_auc = roc_auc_score(valid_y_epoch, valid_prob_epoch)
            valid_threshold_epoch, _ = select_threshold(valid_y_epoch, valid_prob_epoch)
            valid_metrics_epoch = evaluate_probabilities(
                valid_y_epoch, valid_prob_epoch, valid_threshold_epoch
            )
            torch_history_rows.append(
                {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_auc": valid_auc,
                    "val_f1": valid_metrics_epoch["F1"],
                    "val_recall": valid_metrics_epoch["Recall"],
                }
            )
            print(
                f"Epoch {epoch:02d} - train_loss={train_loss:.4f} - val_auc={valid_auc:.4f} - val_f1={valid_metrics_epoch['F1']:.4f}"
            )

            if valid_auc > best_auc:
                best_auc = valid_auc
                best_state = {
                    k: v.detach().cpu().clone() for k, v in torch_model.state_dict().items()
                }
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= PATIENCE:
                    print("Early stopping triggered.")
                    break

        torch_train_time = time.perf_counter() - start
        torch_model.load_state_dict(best_state)
        torch_checkpoint = MODEL_DIR / "pytorch_basic_mlp_best.pt"
        torch.save(
            {
                "model_state_dict": torch_model.state_dict(),
                "input_dim": input_dim,
                "feature_columns": feature_cols,
                "seed": SEED,
            },
            torch_checkpoint,
        )
        torch_history = pd.DataFrame(torch_history_rows)
        torch_history.to_csv(METRIC_DIR / "pytorch_basic_mlp_history.csv", index=False)
        print(f"PyTorch train time: {torch_train_time:.2f} seconds")
        display(torch_history)

Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ các bước `zero_grad`, forward, loss, backward và optimizer step. Checkpoint tốt nhất được chọn bằng validation ROC-AUC, không dùng test set.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng protocol như Keras.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        valid_y_torch, torch_val_prob = torch_predict_prob(torch_model, valid_loader)
        torch_threshold, torch_threshold_table = select_threshold(valid_y_torch, torch_val_prob)
        torch_metrics = evaluate_probabilities(valid_y_torch, torch_val_prob, torch_threshold)
        torch_report = classification_report(
            valid_y_torch.astype(int),
            (torch_val_prob >= torch_threshold).astype(int),
            output_dict=True,
            zero_division=0,
        )

        torch_result = {
            "Model": "M1 Basic MLP",
            "Framework": "PyTorch",
            **torch_metrics,
            "Params": int(count_torch_params(torch_model)),
            "Train Time": float(torch_train_time),
            "Checkpoint": torch_checkpoint.relative_to(PROJECT_ROOT).as_posix(),
        }

        save_json(METRIC_DIR / "pytorch_basic_mlp_validation_metrics.json", torch_result)
        torch_threshold_table.to_csv(
            METRIC_DIR / "pytorch_basic_mlp_thresholds.csv", index=False
        )
        pd.DataFrame(torch_report).T.to_csv(
            METRIC_DIR / "pytorch_basic_mlp_classification_report.csv"
        )

        display(pd.DataFrame([torch_result]).drop(columns=["Checkpoint"]))
        display(
            pd.DataFrame(
                confusion_matrix(
                    valid_y_torch, (torch_val_prob >= torch_threshold).astype(int)
                ),
                index=["Actual 0", "Actual 1"],
                columns=["Pred 0", "Pred 1"],
            )
        )
        display(pd.DataFrame(torch_report).T.round(4))

Output cho thấy điều gì?

PyTorch được đánh giá trên cùng validation set và cùng cách chọn threshold theo F1. Vì framework, initialization và optimizer implementation có khác biệt nhỏ, một run không đủ để kết luận framework nào tốt hơn về bản chất.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng cùng metric validation.


Output cho thấy điều gì?

Bảng chỉ so sánh hai implementation baseline trong điều kiện hiện tại. Chênh lệch nhỏ có thể đến từ initialization, training loop và numerical behavior; không nên kết luận framework nào intrinsically better từ một run.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture MLP phù hợp dữ liệu tabular: M1 Basic MLP, M2 BatchNorm MLP, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được ép vào dữ liệu BRFSS vì feature order không có local structure tự nhiên.


Output cho thấy điều gì?

Các experiment giữ cùng split, preprocessing, seed và training budget. CNN được xử lý như phần discovery/toy operation, không đưa vào model chính vì không có cơ sở spatial/sequential trên raw representation.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation metrics và chọn ứng viên tốt nhất bằng validation F1.


Output cho thấy điều gì?

Bảng xếp hạng theo validation F1 vì dataset mất cân bằng và chỉ accuracy sẽ thiên về class 0. Trade-off cần đọc cùng Precision/Recall: recall cao hơn thường giảm false negatives nhưng có thể tăng false positives.


# 19. Representation Inspection

Cell này làm gì?

Inspect representation bên trong: activation của hidden layer cho model tốt nhất và attention weights nếu M4 được train.


Output cho thấy điều gì?

Hidden activations cho biết tabular vector được model biến đổi thành representation dày hơn trước output. Nếu có attention summary, đó là inspection phụ trợ về trọng số tương đối của feature trong M4, không phải bằng chứng nhân quả y khoa.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi trên validation set: false positives, false negatives và high-confidence errors của model được chọn theo validation.


Output cho thấy điều gì?

False negatives là nhóm cần chú ý trong bài toán diabetes vì model bỏ sót class 1 trên validation. High-confidence errors giúp nhận diện các vùng dữ liệu model tự tin nhưng sai; các nhận xét này chỉ dùng cho phân tích mô hình, không phải khuyến nghị lâm sàng.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation, evaluate test set đúng một lần, rồi lưu final model và metrics.


Output cho thấy điều gì?

Đây là kết quả test cuối cùng của model đã chọn bằng validation. `M3 Residual MLP` dùng threshold 0.63; test Accuracy 0.7844, Precision 0.3670, Recall 0.6084, F1 0.4578, ROC-AUC 0.8089.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc artifacts A03 nếu có và so sánh dataset/workflow/metrics với A04. Không kết luận A04 tốt hơn chỉ vì score khác, vì dataset đã đổi.


Output cho thấy điều gì?

A03 và A04 cùng bài toán nhưng khác dataset: A03 cân bằng hơn, A04 thực tế hơn và mất cân bằng hơn. Vì vậy metric không chứng minh deep learning intrinsically better/worse; nó chỉ mô tả workflow và kết quả trên dữ liệu A04 đã chọn.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


Output cho thấy điều gì?

Bảng cuối ghi lại dataset, model được chọn, threshold và test metrics thật. Artifact listing xác nhận notebook đã lưu model, metrics, figures và metadata cho phase Diabetes A04; recorded training/evaluation time khoảng 306.08 giây.


## Thực hành: house_price_keras

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

# A04 house_price — keras

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Housing Price: dùng đặc trưng property để dự đoán giá nhà liên tục. A04 dùng King County House Sales trong `datasets/housing_price/a04`, tập trung vào deep learning regression trên dữ liệu tabular và so sánh cẩn thận với A03.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        from pathlib import Path
        import json
        import pandas as pd

        def find_project_root(start=None):
            start = Path.cwd() if start is None else Path(start)
            for path in [start.resolve(), *start.resolve().parents]:
                if (path / "datasets" / "housing_price" / "a04").exists() and (path / "pipeline").exists():
                    return path
            raise FileNotFoundError("Cannot locate project root from current working directory.")

        PROJECT_ROOT = find_project_root()
        DATA_DIR = PROJECT_ROOT / "datasets" / "housing_price" / "a04"
        A03_NOTEBOOK = PROJECT_ROOT / "pipeline" / "house_price_pipeline.ipynb"
        A03_METADATA_PATH = PROJECT_ROOT / "pipeline" / "house_price" / "metadata.json"
        A03_COMPARISON_PATH = PROJECT_ROOT / "pipeline" / "house_price" / "final_comparison.csv"
        A04_METADATA_PATH = DATA_DIR / "metadata.json"

        with A04_METADATA_PATH.open("r", encoding="utf-8") as f:
            dataset_metadata = json.load(f)

        overview = {
            "A03 task": "Regression: housing/property features -> house price",
            "A03 target": "Price_BillionVND",
            "A03 notebook exists": A03_NOTEBOOK.exists(),
            "A04 dataset": dataset_metadata["dataset_name"],
            "A04 source": dataset_metadata["source"],
            "A04 target": dataset_metadata["target"],
            "A04 samples": dataset_metadata["samples"],
            "A04 features": dataset_metadata["features"],
            "Task type": "Regression",
        }
        pd.DataFrame([overview]).T.rename(columns={0: "Value"})


Output cho thấy điều gì?

Bảng xác nhận A04 giữ đúng bài toán regression của A03: đầu vào là feature mô tả nhà/property, target là giá bán liên tục. Dataset A04 đổi sang King County nên metric cuối chỉ so sánh workflow và chất lượng dữ liệu, không dùng như benchmark trực tiếp với A03.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để path portable trong repo.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    import os
    import random
    import time
    import warnings
    from datetime import datetime, timezone
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import sklearn
    from IPython.display import display
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    warnings.filterwarnings('ignore', category=FutureWarning)
    SEED = 42
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
    tf.get_logger().setLevel('ERROR')
    tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
    ARTIFACT_DIR = PROJECT_ROOT / 'pipeline04' / 'house_price'
    MODEL_DIR = ARTIFACT_DIR / 'models'
    METRIC_DIR = ARTIFACT_DIR / 'metrics'
    FIGURE_DIR = ARTIFACT_DIR / 'figures'
    for directory in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'TensorFlow': tf.__version__, 'TensorFlow GPUs': len(tf.config.list_physical_devices('GPU')), 'Project root is portable': PROJECT_ROOT.name == 'assign'}
    pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


Output cho thấy điều gì?

Environment có đủ hai framework để chạy cùng một dữ liệu đã preprocess. CPU vẫn phù hợp vì King County có khoảng vài chục nghìn dòng, nhỏ hơn nhiều so với các bài toán ảnh/video.


# 3. Load Dataset

Cell này làm gì?

Đọc raw CSV A04 từ `datasets/housing_price/a04`. File gốc được giữ nguyên; mọi xử lý nằm trong DataFrame và pipeline preprocessing.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        csv_files = sorted(DATA_DIR.glob("*.csv"))
        raw_files = [
            {
                "file": path.relative_to(PROJECT_ROOT).as_posix(),
                "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
            }
            for path in csv_files
        ]
        display(pd.DataFrame(raw_files))

        DATA_PATH = DATA_DIR / "kc_house_data.csv"
        TARGET = "price"
        df_raw = pd.read_csv(DATA_PATH)

        print("Loaded dataset:", DATA_PATH.relative_to(PROJECT_ROOT).as_posix())
        print("Shape:", df_raw.shape)
        display(df_raw.head())


Output cho thấy điều gì?

Dataset được đọc từ path tương đối trong project. `price` là target regression liên tục; các cột còn lại mô tả thời gian bán, diện tích, vị trí, tình trạng và đặc điểm property.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra schema, missing values, duplicates, phân phối target, descriptive statistics và các giá trị housing bất thường.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        shape_info = pd.DataFrame(
            {
                "Metric": ["Rows", "Columns", "Candidate feature columns"],
                "Value": [df_raw.shape[0], df_raw.shape[1], df_raw.shape[1] - 1],
            }
        )
        dtype_info = df_raw.dtypes.astype(str).reset_index()
        dtype_info.columns = ["Column", "Dtype"]

        missing = df_raw.isna().sum().reset_index()
        missing.columns = ["Column", "MissingCount"]
        missing["MissingRate"] = missing["MissingCount"] / len(df_raw)
        duplicate_count = int(df_raw.duplicated().sum())
        property_duplicate_count = int(df_raw['id'].duplicated().sum()) if 'id' in df_raw.columns else 0

        price_stats = df_raw[TARGET].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_frame("price").T
        price_skew = float(df_raw[TARGET].skew())
        q1, q3 = df_raw[TARGET].quantile([0.25, 0.75])
        iqr = q3 - q1
        outlier_upper = q3 + 1.5 * iqr
        price_outliers = int((df_raw[TARGET] > outlier_upper).sum())

        quality_checks = pd.DataFrame(
            {
                "Check": [
                    "price <= 0",
                    "sqft_living <= 0",
                    "sqft_lot <= 0",
                    "bedrooms == 0",
                    "bedrooms > 15",
                    "bathrooms == 0",
                    "yr_built <= 0",
                ],
                "Count": [
                    int((df_raw[TARGET] <= 0).sum()),
                    int((df_raw["sqft_living"] <= 0).sum()),
                    int((df_raw["sqft_lot"] <= 0).sum()),
                    int((df_raw["bedrooms"] == 0).sum()),
                    int((df_raw["bedrooms"] > 15).sum()),
                    int((df_raw["bathrooms"] == 0).sum()),
                    int((df_raw["yr_built"] <= 0).sum()),
                ],
            }
        )

        display(shape_info)
        display(dtype_info)
        display(missing[missing["MissingCount"] > 0] if missing["MissingCount"].sum() else pd.DataFrame({"Missing": ["No missing values detected"]}))
        print("Duplicate rows:", duplicate_count)
        print("Repeated property id rows:", property_duplicate_count)
        print("Price skew:", round(price_skew, 4))
        print("IQR upper outlier boundary:", round(float(outlier_upper), 2))
        print("Price IQR outlier count:", price_outliers)
        display(price_stats.round(2))
        display(df_raw.describe().T.round(2))
        display(quality_checks)

        fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
        axes[0].hist(df_raw[TARGET], bins=60, color="#4C78A8")
        axes[0].set_title("Price Distribution")
        axes[0].set_xlabel("price")
        axes[0].set_ylabel("count")
        axes[1].hist(np.log1p(df_raw[TARGET]), bins=60, color="#F58518")
        axes[1].set_title("log1p(price) Distribution")
        axes[1].set_xlabel("log1p(price)")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "target_price_distribution.png", dpi=160)
        plt.show()


Output cho thấy điều gì?

Dataset gốc không có missing value và không có full duplicate rows, nhưng có repeated property `id` do một số nhà được bán nhiều lần. Đây là rủi ro leakage nếu cùng một property xuất hiện ở nhiều split, nên bước chuẩn bị sẽ giữ latest sale per `id` trước khi chia train/validation/test. Phân phối `price` lệch phải rõ rệt, vì vậy notebook dùng log1p target cho training và inverse transform khi báo metrics trên price scale gốc.


# 5. Data Preparation

Cell này làm gì?

Tạo feature engineering từ `date`, loại identifier `id`, chia train/validation/test theo price bins, fit preprocessing chỉ trên train, và quyết định target log-transform dựa trên skew.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def add_housing_features(df):
            data = df.copy()
            sale_date = pd.to_datetime(data["date"].astype(str).str.slice(0, 8), format="%Y%m%d", errors="coerce")
            data["sale_year"] = sale_date.dt.year
            data["sale_month"] = sale_date.dt.month
            data["house_age_at_sale"] = (data["sale_year"] - data["yr_built"]).clip(lower=0)
            data["renovated_flag"] = (data["yr_renovated"] > 0).astype(int)
            data["years_since_renovation"] = np.where(
                data["yr_renovated"] > 0,
                (data["sale_year"] - data["yr_renovated"]).clip(lower=0),
                0,
            )
            data["basement_flag"] = (data["sqft_basement"] > 0).astype(int)
            data["zipcode"] = data["zipcode"].astype(str)
            return data

        df_model = df_raw.drop_duplicates().copy()
        repeated_property_rows_removed = int(df_model['id'].duplicated().sum())
        df_model = df_model.sort_values('date').drop_duplicates(subset=['id'], keep='last')
        df_model = df_model[df_model[TARGET] > 0].reset_index(drop=True)
        df_model = add_housing_features(df_model)

        numeric_features = [
            "bedrooms",
            "bathrooms",
            "sqft_living",
            "sqft_lot",
            "floors",
            "waterfront",
            "view",
            "condition",
            "grade",
            "sqft_above",
            "sqft_basement",
            "yr_built",
            "yr_renovated",
            "lat",
            "long",
            "sqft_living15",
            "sqft_lot15",
            "sale_year",
            "sale_month",
            "house_age_at_sale",
            "renovated_flag",
            "years_since_renovation",
            "basement_flag",
        ]
        categorical_features = ["zipcode"]
        feature_cols = numeric_features + categorical_features

        X = df_model[feature_cols]
        y = df_model[TARGET].astype(float)
        price_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")

        X_train_raw, X_temp_raw, y_train, y_temp, bins_train, bins_temp = train_test_split(
            X,
            y,
            price_bins,
            test_size=0.30,
            random_state=SEED,
            stratify=price_bins,
        )
        X_valid_raw, X_test_raw, y_valid, y_test = train_test_split(
            X_temp_raw,
            y_temp,
            test_size=0.50,
            random_state=SEED,
            stratify=bins_temp,
        )

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="median")),
                            ("scaler", StandardScaler()),
                        ]
                    ),
                    numeric_features,
                ),
                (
                    "categorical",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="most_frequent")),
                            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                        ]
                    ),
                    categorical_features,
                ),
            ],
            sparse_threshold=0.0,
        )

        X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
        X_valid = preprocessor.transform(X_valid_raw).astype("float32")
        X_test = preprocessor.transform(X_test_raw).astype("float32")
        joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")

        y_train_np = y_train.to_numpy(dtype="float32")
        y_valid_np = y_valid.to_numpy(dtype="float32")
        y_test_np = y_test.to_numpy(dtype="float32")
        target_skew_train = float(pd.Series(y_train_np).skew())
        USE_LOG_TARGET = target_skew_train > 1.0

        def transform_target(y_values):
            values = np.log1p(y_values) if USE_LOG_TARGET else y_values
            return ((values - target_center) / target_scale).astype("float32")

        def inverse_transform_target(y_scaled):
            values = np.asarray(y_scaled).reshape(-1) * target_scale + target_center
            if USE_LOG_TARGET:
                return np.expm1(values)
            return values

        target_train_base = np.log1p(y_train_np) if USE_LOG_TARGET else y_train_np
        target_center = float(target_train_base.mean())
        target_scale = float(target_train_base.std())
        y_train_model = transform_target(y_train_np)
        y_valid_model = transform_target(y_valid_np)
        y_test_model = transform_target(y_test_np)

        transformed_feature_names = preprocessor.get_feature_names_out()
        split_summary = pd.DataFrame(
            [
                {"Split": "train", "Rows": len(y_train), "MeanPrice": y_train.mean(), "MedianPrice": y_train.median()},
                {"Split": "validation", "Rows": len(y_valid), "MeanPrice": y_valid.mean(), "MedianPrice": y_valid.median()},
                {"Split": "test", "Rows": len(y_test), "MeanPrice": y_test.mean(), "MedianPrice": y_test.median()},
            ]
        )

        target_transform_summary = pd.DataFrame(
            [
                {
                    "TrainPriceSkew": target_skew_train,
                    "UseLog1pTarget": USE_LOG_TARGET,
                    "TargetCenter": target_center,
                    "TargetScale": target_scale,
                    "RawFeatures": len(feature_cols),
                    "TransformedFeatures": X_train.shape[1],
                }
            ]
        )

        print("Rows before duplicate removal:", len(df_raw))
        print("Rows after duplicate removal and valid target filter:", len(df_model))
        display(split_summary.round(2))
        display(target_transform_summary.round(4))
        display(pd.DataFrame({"Feature": transformed_feature_names[:20]}))


Output cho thấy điều gì?

Preprocessing bắt đầu bằng cách bỏ full duplicates, loại `price <= 0` nếu có, và giữ latest sale per property `id` để tránh cùng một property đi qua nhiều split. Split được stratify theo price bins; preprocessor chỉ fit trên train set, còn validation/test chỉ dùng `transform`, nên scaling/encoding không học từ held-out data.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của dữ liệu deep learning và đánh giá CNN suitability cho housing-price tabular regression.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        representation_summary = pd.DataFrame(
            {
                "Array": ["X_train", "y_train_model", "X_valid", "y_valid_model", "X_test", "y_test_model"],
                "Shape": [X_train.shape, y_train_model.shape, X_valid.shape, y_valid_model.shape, X_test.shape, y_test_model.shape],
                "Meaning": [
                    "(samples, transformed features)",
                    "(samples,)",
                    "(samples, transformed features)",
                    "(samples,)",
                    "(samples, transformed features)",
                    "(samples,)",
                ],
            }
        )
        cnn_suitability = pd.DataFrame(
            [
                {"Question": "Raw representation", "Assessment": "Tabular property vector with numeric and one-hot zipcode features"},
                {"Question": "Natural baseline", "Assessment": "MLP/DNN regression because each house is a fixed feature vector"},
                {"Question": "CNN justified?", "Assessment": "No production CNN: columns do not form an image or meaningful local sequence"},
                {"Question": "CNN usage in this notebook", "Assessment": "Only toy Conv1D operations for understanding convolution, not final modeling"},
            ]
        )
        display(representation_summary)
        display(cnn_suitability)


Output cho thấy điều gì?

`X_train` là ma trận `(samples, transformed features)`, trong đó zipcode đã được one-hot encode và numeric features đã scale. Đây là vector tabular; MLP regression là lựa chọn tự nhiên hơn CNN.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa Dense/Linear, ReLU, output regression, Conv1D và pooling bằng toy tensors.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        toy_x = tf.constant([[1200.0, 3.0, 2.0]], dtype=tf.float32)
        dense = keras.Sequential(
            [
                layers.Input(shape=(3,)),
                layers.Dense(2, use_bias=True, kernel_initializer="ones", bias_initializer="zeros"),
                layers.ReLU(),
                layers.Dense(1, use_bias=True, kernel_initializer="ones", bias_initializer="zeros"),
            ]
        )
        toy_regression_output = dense(toy_x)

        toy_seq = tf.reshape(tf.range(1, 9, dtype=tf.float32), (1, 8, 1))
        conv = layers.Conv1D(filters=2, kernel_size=3, padding="valid", activation="relu")
        conv_out = conv(toy_seq)
        pool_out = layers.MaxPooling1D(pool_size=2)(conv_out)

        operation_shapes = pd.DataFrame(
            [
                {"Object": "toy_x", "Shape": tuple(toy_x.shape), "Meaning": "one toy property vector"},
                {"Object": "Regression output", "Shape": tuple(toy_regression_output.shape), "Meaning": "one scalar continuous prediction"},
                {"Object": "Toy sequence", "Shape": tuple(toy_seq.shape), "Meaning": "(batch, steps, channels)"},
                {"Object": "Conv1D output", "Shape": tuple(conv_out.shape), "Meaning": "local windows over an ordered toy sequence"},
                {"Object": "Pooling output", "Shape": tuple(pool_out.shape), "Meaning": "downsampled convolution features"},
            ]
        )
        display(operation_shapes)
        print("Toy scalar output:", np.round(toy_regression_output.numpy(), 4))


Output cho thấy điều gì?

Regression output là một scalar tuyến tính, không phải xác suất hay class label. Conv1D/pooling chỉ minh họa khi dữ liệu có trục thứ tự; King County sau preprocessing không có cấu trúc cục bộ như vậy.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, forward network, output regression, MSE/MAE helper và Conv1D toy.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def linear_np(x, w, b):
            return x @ w + b

        def relu_np(x):
            return np.maximum(0, x)

        def mse_np(y_true, y_pred):
            return np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)

        def mae_np(y_true, y_pred):
            return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))

        def forward_regression_np(x, params):
            hidden = relu_np(linear_np(x, params["w1"], params["b1"]))
            return linear_np(hidden, params["w2"], params["b2"]).ravel()

        def conv1d_valid_np(sequence, kernel, bias=0.0):
            sequence = np.asarray(sequence, dtype=float)
            kernel = np.asarray(kernel, dtype=float)
            out = []
            for i in range(len(sequence) - len(kernel) + 1):
                out.append(float(np.sum(sequence[i : i + len(kernel)] * kernel) + bias))
            return np.asarray(out)

        rng = np.random.default_rng(SEED)
        toy_batch = X_train[:4]
        params = {
            "w1": rng.normal(0, 0.1, size=(X_train.shape[1], 8)),
            "b1": np.zeros(8),
            "w2": rng.normal(0, 0.1, size=(8, 1)),
            "b2": np.zeros(1),
        }
        toy_scaled_pred = forward_regression_np(toy_batch, params)
        toy_price_pred = inverse_transform_target(toy_scaled_pred)
        toy_conv = conv1d_valid_np([1, 2, 3, 4, 5], [0.25, 0.5, 0.25])

        print("Toy scaled predictions:", np.round(toy_scaled_pred, 4))
        print("Toy price predictions:", np.round(toy_price_pred, 2))
        print("Toy MSE on scaled target:", round(float(mse_np(y_train_model[:4], toy_scaled_pred)), 4))
        print("Toy MAE on scaled target:", round(float(mae_np(y_train_model[:4], toy_scaled_pred)), 4))
        print("Manual Conv1D toy output:", np.round(toy_conv, 4))


Output cho thấy điều gì?

NumPy forward pass minh họa cách MLP regression tạo một giá trị liên tục. MSE/MAE được tính trên target đã transform trong toy example; khi đánh giá model thật, notebook chuyển prediction về price scale gốc.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric regression và baseline Keras MLP với output scalar tuyến tính.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def to_jsonable(obj):
            if isinstance(obj, dict):
                return {str(k): to_jsonable(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [to_jsonable(v) for v in obj]
            if isinstance(obj, tuple):
                return [to_jsonable(v) for v in obj]
            if isinstance(obj, (np.integer,)):
                return int(obj)
            if isinstance(obj, (np.floating,)):
                return float(obj)
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            return obj

        def save_json(path, obj):
            with Path(path).open("w", encoding="utf-8") as f:
                json.dump(to_jsonable(obj), f, indent=2)

        def regression_metrics(y_true_price, y_pred_price):
            y_pred_price = np.maximum(np.asarray(y_pred_price).reshape(-1), 0)
            mse = mean_squared_error(y_true_price, y_pred_price)
            return {
                "MAE": float(mean_absolute_error(y_true_price, y_pred_price)),
                "MSE": float(mse),
                "RMSE": float(np.sqrt(mse)),
                "R2": float(r2_score(y_true_price, y_pred_price)),
            }

        def predict_price_keras(model, X_data):
            scaled_pred = model.predict(X_data, batch_size=BATCH_SIZE, verbose=0).ravel()
            return np.maximum(inverse_transform_target(scaled_pred), 0)

        def compile_regression_model(model):
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                loss=keras.losses.Huber(delta=1.0),
                metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
            )
            return model

        def build_basic_mlp(input_dim, name="M1_Basic_MLP_Regressor"):
            inputs = keras.Input(shape=(input_dim,), name="property_feature_vector")
            x = layers.Dense(128, activation="relu", name="dense_128")(inputs)
            x = layers.Dense(64, activation="relu", name="dense_64")(x)
            outputs = layers.Dense(1, activation="linear", name="price_output")(x)
            return compile_regression_model(keras.Model(inputs, outputs, name=name))

        BATCH_SIZE = 512
        MAX_EPOCHS = 30
        PATIENCE = 5
        input_dim = X_train.shape[1]

        keras_baseline = build_basic_mlp(input_dim)
        keras_baseline.summary()
        print("Input shape:", X_train.shape)
        print("Output shape for 5 samples:", tuple(keras_baseline(X_train[:5]).shape))
        print("Total parameters:", keras_baseline.count_params())


Output cho thấy điều gì?

Keras baseline nhận vector property đã transform và xuất một scalar linear trên target đã chuẩn hóa. Loss Huber phù hợp với target có skew/outliers vì bớt nhạy với sai số cực lớn hơn MSE thuần.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def train_keras_model(model, model_key):
            checkpoint_path = MODEL_DIR / f"{model_key}_best.keras"
            callbacks = [
                keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=PATIENCE, restore_best_weights=True),
                keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", mode="min", save_best_only=True),
            ]
            start = time.perf_counter()
            history = model.fit(
                X_train,
                y_train_model,
                validation_data=(X_valid, y_valid_model),
                epochs=MAX_EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=callbacks,
                verbose=2,
            )
            train_time = time.perf_counter() - start
            history_df = pd.DataFrame(history.history)
            history_df.to_csv(METRIC_DIR / f"{model_key}_history.csv", index=False)
            return history, train_time, checkpoint_path

        keras_history, keras_train_time, keras_checkpoint = train_keras_model(keras_baseline, "m1_basic_mlp_regressor")
        print(f"Keras baseline train time: {keras_train_time:.2f} seconds")
        print("Best checkpoint:", keras_checkpoint.relative_to(PROJECT_ROOT).as_posix())
        display(pd.DataFrame(keras_history.history).tail())


Output cho thấy điều gì?

Training log hiển thị loss và MAE trên target đã transform cho train/validation. Checkpoint chọn epoch có validation loss thấp nhất; test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và MAE của baseline Keras trên train/validation, đồng thời lưu figure.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def plot_history(history, title, output_path):
            history_df = pd.DataFrame(history.history)
            fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
            axes[0].plot(history_df["loss"], label="train")
            axes[0].plot(history_df["val_loss"], label="validation")
            axes[0].set_title(f"{title} Loss")
            axes[0].set_xlabel("Epoch")
            axes[0].legend()
            axes[1].plot(history_df["mae"], label="train")
            axes[1].plot(history_df["val_mae"], label="validation")
            axes[1].set_title(f"{title} MAE on transformed target")
            axes[1].set_xlabel("Epoch")
            axes[1].legend()
            fig.tight_layout()
            fig.savefig(output_path, dpi=160)
            plt.show()

        plot_history(keras_history, "Keras Basic MLP Regressor", FIGURE_DIR / "keras_basic_mlp_training_curves.png")


Output cho thấy điều gì?

Curves cho biết model học ổn định hay overfit trên validation. Vì target đã log/scale, magnitude của loss dùng để chọn model; metric giá nhà dễ hiểu hơn được tính riêng sau khi inverse transform.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng MAE, MSE, RMSE, R²; lưu actual-vs-predicted, residual plot và error distribution.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        keras_val_pred = predict_price_keras(keras_baseline, X_valid)
        keras_metrics = regression_metrics(y_valid_np, keras_val_pred)
        keras_result = {
            "Model": "M1 Basic MLP Regressor",
            "Framework": "Keras",
            **keras_metrics,
            "Params": int(keras_baseline.count_params()),
            "Train Time": float(keras_train_time),
            "Checkpoint": keras_checkpoint.relative_to(PROJECT_ROOT).as_posix(),
        }
        save_json(METRIC_DIR / "keras_basic_mlp_validation_metrics.json", keras_result)

        residuals = y_valid_np - keras_val_pred
        fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
        axes[0].scatter(y_valid_np, keras_val_pred, s=8, alpha=0.35, color="#4C78A8")
        lims = [min(y_valid_np.min(), keras_val_pred.min()), max(y_valid_np.max(), keras_val_pred.max())]
        axes[0].plot(lims, lims, "--", color="gray")
        axes[0].set_title("Actual vs Predicted")
        axes[0].set_xlabel("Actual price")
        axes[0].set_ylabel("Predicted price")
        axes[1].scatter(keras_val_pred, residuals, s=8, alpha=0.35, color="#F58518")
        axes[1].axhline(0, color="gray", linestyle="--")
        axes[1].set_title("Residual Plot")
        axes[1].set_xlabel("Predicted price")
        axes[1].set_ylabel("Actual - Predicted")
        axes[2].hist(residuals, bins=50, color="#54A24B")
        axes[2].set_title("Residual Distribution")
        axes[2].set_xlabel("Residual")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "keras_validation_regression_diagnostics.png", dpi=160)
        plt.show()

        display(pd.DataFrame([keras_result]).drop(columns=["Checkpoint"]).round(4))


Output cho thấy điều gì?

Keras baseline validation đạt MAE 70,219, RMSE 138,387, R² 0.8679. Diagnostic plots cho thấy sai số vẫn lớn hơn ở vùng giá cao, phù hợp với target có đuôi phải dài.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước train, chạy một batch qua model để kiểm tra shape và parameter count.


Output cho thấy điều gì?

Batch input có dạng `(batch, transformed features)` và output có dạng `(batch, 1)`, đúng với regression scalar. Kiến trúc PyTorch tương đương ý tưởng với Keras baseline để so sánh framework công bằng hơn.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline bằng loop thủ công, dùng SmoothL1Loss/Huber và lưu checkpoint tốt nhất theo validation loss.


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ `zero_grad`, forward, loss, backward và optimizer step. Validation RMSE/R² trong log đã inverse transform về price scale để dễ đọc hơn loss nội bộ.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng metric price-scale như Keras.


Output cho thấy điều gì?

PyTorch baseline được đánh giá trên cùng validation split và cùng target inverse transform. Metric khác biệt giữa hai framework có thể đến từ initialization và training loop, không đủ để kết luận framework nào tốt hơn về bản chất.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng validation MAE/RMSE/R².


Output cho thấy điều gì?

Trong baseline framework comparison, PyTorch có validation RMSE thấp nhất (113,519) ở run này. Đây chỉ là kết quả một lần chạy với cùng ý tưởng architecture, không phải kết luận framework nào tốt hơn về bản chất.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture phù hợp dữ liệu tabular: M1 Basic MLP Regressor, M2 BatchNorm, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được train như production model vì feature order không có ý nghĩa không gian.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        def build_batchnorm_mlp(input_dim, name="M2_BatchNorm_MLP_Regressor"):
            inputs = keras.Input(shape=(input_dim,), name="property_feature_vector")
            x = layers.Dense(128, use_bias=False, name="dense_128")(inputs)
            x = layers.BatchNormalization(name="bn_128")(x)
            x = layers.Activation("relu", name="relu_128")(x)
            x = layers.Dense(64, use_bias=False, name="dense_64")(x)
            x = layers.BatchNormalization(name="bn_64")(x)
            x = layers.Activation("relu", name="relu_64")(x)
            outputs = layers.Dense(1, activation="linear", name="price_output")(x)
            return compile_regression_model(keras.Model(inputs, outputs, name=name))

        def residual_block(x, units, block_name):
            shortcut = x
            y = layers.Dense(units, activation="relu", name=f"{block_name}_dense_a")(x)
            y = layers.Dense(units, name=f"{block_name}_dense_b")(y)
            x = layers.Add(name=f"{block_name}_add")([shortcut, y])
            return layers.Activation("relu", name=f"{block_name}_relu")(x)

        def build_residual_mlp(input_dim, name="M3_Residual_MLP_Regressor"):
            inputs = keras.Input(shape=(input_dim,), name="property_feature_vector")
            x = layers.Dense(128, activation="relu", name="input_projection")(inputs)
            x = residual_block(x, 128, "residual_1")
            x = residual_block(x, 128, "residual_2")
            x = layers.Dense(64, activation="relu", name="dense_64")(x)
            outputs = layers.Dense(1, activation="linear", name="price_output")(x)
            return compile_regression_model(keras.Model(inputs, outputs, name=name))

        def build_attention_residual_mlp(input_dim, name="M4_Residual_MLP_Feature_Attention_Regressor"):
            inputs = keras.Input(shape=(input_dim,), name="property_feature_vector")
            attention = layers.Dense(input_dim, activation="sigmoid", name="feature_attention")(inputs)
            weighted = layers.Multiply(name="attention_weighted_features")([inputs, attention])
            x = layers.Dense(128, activation="relu", name="input_projection")(weighted)
            x = residual_block(x, 128, "residual_1")
            x = residual_block(x, 128, "residual_2")
            x = layers.Dense(64, activation="relu", name="dense_64")(x)
            outputs = layers.Dense(1, activation="linear", name="price_output")(x)
            return compile_regression_model(keras.Model(inputs, outputs, name=name))

        def evaluate_keras_model(model, model_key, checkpoint_path, train_time):
            val_pred = predict_price_keras(model, X_valid)
            metrics = regression_metrics(y_valid_np, val_pred)
            result = {
                "Model": model.name.replace("_", " "),
                "Framework": "Keras",
                **metrics,
                "Params": int(model.count_params()),
                "Train Time": float(train_time),
                "Checkpoint": checkpoint_path.relative_to(PROJECT_ROOT).as_posix(),
            }
            save_json(METRIC_DIR / f"{model_key}_validation_metrics.json", result)
            return result

        architecture_results = [keras_result]
        architecture_histories = {"M1 Basic MLP Regressor": keras_history}
        trained_keras_models = {"M1 Basic MLP Regressor": keras_baseline}

        for model_key, builder in [
            ("m2_batchnorm_mlp_regressor", build_batchnorm_mlp),
            ("m3_residual_mlp_regressor", build_residual_mlp),
            ("m4_attention_residual_mlp_regressor", build_attention_residual_mlp),
        ]:
            tf.random.set_seed(SEED)
            model = builder(input_dim)
            print("\nTraining", model.name, "with", model.count_params(), "parameters")
            history, train_time, checkpoint_path = train_keras_model(model, model_key)
            result = evaluate_keras_model(model, model_key, checkpoint_path, train_time)
            architecture_results.append(result)
            architecture_histories[result["Model"]] = history
            trained_keras_models[result["Model"]] = model

        cnn_investigation = pd.DataFrame(
            [
                {
                    "Investigation": "CNN on King County tabular features",
                    "Decision": "Not trained as a candidate model",
                    "Reason": "Numeric + one-hot property columns do not have meaningful local order; Conv1D/Conv2D would add artificial structure.",
                }
            ]
        )
        display(cnn_investigation)


Output cho thấy điều gì?

Các architecture đều giữ cùng split, preprocessing, target transform, seed và training budget. CNN được ghi nhận ở mức methodological investigation, không ép thành model vì bài toán là tabular regression.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation MAE/RMSE/R² và chọn ứng viên tốt nhất bằng validation RMSE thấp nhất.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        architecture_comparison = pd.DataFrame(architecture_results)[
            ["Model", "MAE", "MSE", "RMSE", "R2", "Params", "Train Time", "Checkpoint"]
        ].sort_values("RMSE", ascending=True)
        architecture_comparison.to_csv(METRIC_DIR / "architecture_comparison.csv", index=False)

        display(architecture_comparison.drop(columns=["Checkpoint"]).round(4))

        fig, ax = plt.subplots(figsize=(8, 3.8))
        plot_df = architecture_comparison.sort_values("RMSE", ascending=False)
        ax.barh(plot_df["Model"], plot_df["RMSE"], color="#4C78A8")
        ax.set_xlabel("Validation RMSE")
        ax.set_title("A04 Housing Architecture Comparison")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "architecture_comparison_rmse.png", dpi=160)
        plt.show()

        best_validation_row = architecture_comparison.iloc[0].to_dict()
        print("Best validation model:", best_validation_row["Model"])
        print("Best validation RMSE:", round(best_validation_row["RMSE"], 2))


Output cho thấy điều gì?

`M4 Residual MLP Feature Attention Regressor` là model tốt nhất trên validation với RMSE 119,459, MAE 67,889, R² 0.9015. Sau leakage fix repeated property `id`, M4 attention thắng validation RMSE nhưng vẫn cần cân nhắc vì parameter count cao hơn M1.


# 19. Representation Inspection

Cell này làm gì?

Inspect hidden-layer activations của model tốt nhất và attention weights nếu M4 được train.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        best_model_name = best_validation_row["Model"]
        best_checkpoint_path = PROJECT_ROOT / best_validation_row["Checkpoint"]
        best_validation_model = keras.models.load_model(best_checkpoint_path)

        hidden_layer = None
        for layer in best_validation_model.layers:
            if isinstance(layer, layers.Dense) and getattr(layer, "units", None) != 1:
                hidden_layer = layer.name
                break

        if hidden_layer is not None:
            activation_model = keras.Model(best_validation_model.input, best_validation_model.get_layer(hidden_layer).output)
            sample_activations = activation_model.predict(X_valid[:1024], batch_size=1024, verbose=0)
            activation_summary = pd.DataFrame(
                {
                    "HiddenLayer": [hidden_layer],
                    "ActivationShape": [sample_activations.shape],
                    "MeanActivation": [float(sample_activations.mean())],
                    "StdActivation": [float(sample_activations.std())],
                    "ZeroRate": [float((np.abs(sample_activations) < 1e-8).mean())],
                }
            )
            display(activation_summary.round(4))

        attention_model_path = MODEL_DIR / "m4_attention_residual_mlp_regressor_best.keras"
        if attention_model_path.exists():
            attention_model = keras.models.load_model(attention_model_path)
            attention_extractor = keras.Model(attention_model.input, attention_model.get_layer("feature_attention").output)
            attention_values = attention_extractor.predict(X_valid[:2048], batch_size=1024, verbose=0)
            attention_summary = pd.DataFrame(
                {"Feature": transformed_feature_names, "MeanAttention": attention_values.mean(axis=0), "StdAttention": attention_values.std(axis=0)}
            ).sort_values("MeanAttention", ascending=False)
            attention_summary.to_csv(METRIC_DIR / "feature_attention_summary.csv", index=False)
            display(attention_summary.head(12).round(4))
            fig, ax = plt.subplots(figsize=(8, 5))
            top_attention = attention_summary.head(12).sort_values("MeanAttention")
            ax.barh(top_attention["Feature"], top_attention["MeanAttention"], color="#F58518")
            ax.set_title("Top Feature Attention Weights")
            ax.set_xlabel("Mean attention")
            fig.tight_layout()
            fig.savefig(FIGURE_DIR / "feature_attention_weights.png", dpi=160)
            plt.show()


Output cho thấy điều gì?

Hidden activations cho thấy model đã biến đổi vector property thành representation phi tuyến trước output giá. Attention weights, nếu dùng M4, là công cụ inspection model chứ không phải bằng chứng nhân quả về thị trường nhà.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi validation của model được chọn: absolute error lớn nhất, residual theo price range, và nhóm giá thấp/trung/cao.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        best_val_pred = predict_price_keras(best_validation_model, X_valid)
        error_df = X_valid_raw.reset_index(drop=True).copy()
        error_df["actual_price"] = y_valid_np
        error_df["predicted_price"] = best_val_pred
        error_df["residual"] = error_df["actual_price"] - error_df["predicted_price"]
        error_df["absolute_error"] = error_df["residual"].abs()
        error_df["absolute_pct_error"] = error_df["absolute_error"] / error_df["actual_price"].clip(lower=1)
        error_df["price_range"] = pd.qcut(error_df["actual_price"], q=3, labels=["low-price", "mid-price", "high-price"])

        large_errors = error_df.sort_values("absolute_error", ascending=False).head(12)
        range_rows = []
        for name, group in error_df.groupby("price_range", observed=True):
            metrics = regression_metrics(group["actual_price"].to_numpy(), group["predicted_price"].to_numpy())
            range_rows.append({"PriceRange": str(name), "Count": len(group), **metrics})
        range_error = pd.DataFrame(range_rows)

        error_df.to_csv(METRIC_DIR / "validation_error_analysis.csv", index=False)
        large_errors.to_csv(METRIC_DIR / "validation_large_error_samples.csv", index=False)
        range_error.to_csv(METRIC_DIR / "validation_error_by_price_range.csv", index=False)

        display(range_error.round(4))
        display(large_errors[["actual_price", "predicted_price", "absolute_error", "absolute_pct_error", "sqft_living", "bedrooms", "bathrooms", "grade", "zipcode", "lat", "long"]].round(2))

        fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
        axes[0].boxplot([g["absolute_error"] for _, g in error_df.groupby("price_range", observed=True)], tick_labels=["low", "mid", "high"])
        axes[0].set_title("Absolute Error by Price Range")
        axes[0].set_ylabel("Absolute error")
        axes[1].hist(error_df["absolute_error"], bins=60, color="#54A24B")
        axes[1].set_title("Absolute Error Distribution")
        axes[1].set_xlabel("Absolute error")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "validation_error_analysis.png", dpi=160)
        plt.show()


Output cho thấy điều gì?

Validation error lớn nhất nằm ở nhóm high-price với MAE khoảng 118,650 và RMSE khoảng 189,305. Điều này phù hợp với phân phối giá lệch phải: nhà giá cao tạo residual lớn hơn và làm RMSE nhạy hơn MAE.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation RMSE, evaluate test set đúng một lần, rồi lưu final model và diagnostics.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    if RUN_TRAINING:
        final_model = keras.models.load_model(best_checkpoint_path)
        test_pred = predict_price_keras(final_model, X_test)
        final_metrics = regression_metrics(y_test_np, test_pred)

        final_model_path = MODEL_DIR / "final_model.keras"
        final_model.save(final_model_path)

        final_payload = {
            "selected_model": best_validation_row["Model"],
            "selection_metric": "Validation RMSE",
            "test_metrics": final_metrics,
            "final_model_path": final_model_path.relative_to(PROJECT_ROOT).as_posix(),
            "source_checkpoint": best_checkpoint_path.relative_to(PROJECT_ROOT).as_posix(),
            "target_transform": "standardized log1p(price)" if USE_LOG_TARGET else "standardized price",
        }
        save_json(METRIC_DIR / "final_test_metrics.json", final_payload)

        test_residuals = y_test_np - test_pred
        fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
        axes[0].scatter(y_test_np, test_pred, s=8, alpha=0.35, color="#4C78A8")
        lims = [min(y_test_np.min(), test_pred.min()), max(y_test_np.max(), test_pred.max())]
        axes[0].plot(lims, lims, "--", color="gray")
        axes[0].set_title("Final Test Actual vs Predicted")
        axes[0].set_xlabel("Actual price")
        axes[0].set_ylabel("Predicted price")
        axes[1].scatter(test_pred, test_residuals, s=8, alpha=0.35, color="#F58518")
        axes[1].axhline(0, color="gray", linestyle="--")
        axes[1].set_title("Final Test Residual Plot")
        axes[1].set_xlabel("Predicted price")
        axes[1].set_ylabel("Actual - Predicted")
        axes[2].hist(test_residuals, bins=50, color="#54A24B")
        axes[2].set_title("Final Test Residual Distribution")
        axes[2].set_xlabel("Residual")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "final_test_regression_diagnostics.png", dpi=160)
        plt.show()

        display(pd.DataFrame([final_payload["test_metrics"]]).round(4))


Output cho thấy điều gì?

Final test evaluation của `M4 Residual MLP Feature Attention Regressor` đạt MAE 70,282, RMSE 146,275, R² 0.8673 trên price scale gốc sau inverse transform. Test set chỉ dùng ở bước này sau khi chọn model bằng validation RMSE.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc artifacts A03 nếu có và so sánh dataset/workflow/metrics với A04. Dataset khác nhau nên không dùng score difference như benchmark trực tiếp.


Output cho thấy điều gì?

A03 và A04 cùng là housing-price regression nhưng khác thị trường, target scale và feature schema. Vì vậy chênh lệch RMSE/R² không chứng minh model A04 tốt hơn hay kém hơn A03; nó chỉ mô tả kết quả trên dataset A04.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


Output cho thấy điều gì?

Notebook đã lưu final model, preprocessor, comparison metrics, error-analysis CSV và figures. Kết quả cuối: selected model là `M4 Residual MLP Feature Attention Regressor`, test MAE 70,282, RMSE 146,275, R² 0.8673; tổng training/evaluation time được ghi nhận khoảng 82.39 giây.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    from IPython.display import display
    import pandas as pd
    _saved = _repo / 'A04/artifacts/pipeline04/house_price/metrics'
    for _file in sorted(_saved.glob('*comparison*.csv')):
        print(_file.name)
        display(pd.read_csv(_file))


## Thực hành: house_price_torch

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

# A04 house_price — torch

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Housing Price: dùng đặc trưng property để dự đoán giá nhà liên tục. A04 dùng King County House Sales trong `datasets/housing_price/a04`, tập trung vào deep learning regression trên dữ liệu tabular và so sánh cẩn thận với A03.


Output cho thấy điều gì?

Bảng xác nhận A04 giữ đúng bài toán regression của A03: đầu vào là feature mô tả nhà/property, target là giá bán liên tục. Dataset A04 đổi sang King County nên metric cuối chỉ so sánh workflow và chất lượng dữ liệu, không dùng như benchmark trực tiếp với A03.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để path portable trong repo.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    import os
    import random
    import time
    import warnings
    from datetime import datetime, timezone
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import sklearn
    from IPython.display import display
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    warnings.filterwarnings('ignore', category=FutureWarning)
    SEED = 42
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    ARTIFACT_DIR = PROJECT_ROOT / 'pipeline04' / 'house_price'
    MODEL_DIR = ARTIFACT_DIR / 'models'
    METRIC_DIR = ARTIFACT_DIR / 'metrics'
    FIGURE_DIR = ARTIFACT_DIR / 'figures'
    for directory in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'PyTorch': torch.__version__, 'PyTorch device': str(DEVICE), 'Project root is portable': PROJECT_ROOT.name == 'assign'}
    pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


Output cho thấy điều gì?

Environment có đủ hai framework để chạy cùng một dữ liệu đã preprocess. CPU vẫn phù hợp vì King County có khoảng vài chục nghìn dòng, nhỏ hơn nhiều so với các bài toán ảnh/video.


# 3. Load Dataset

Cell này làm gì?

Đọc raw CSV A04 từ `datasets/housing_price/a04`. File gốc được giữ nguyên; mọi xử lý nằm trong DataFrame và pipeline preprocessing.


Output cho thấy điều gì?

Dataset được đọc từ path tương đối trong project. `price` là target regression liên tục; các cột còn lại mô tả thời gian bán, diện tích, vị trí, tình trạng và đặc điểm property.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra schema, missing values, duplicates, phân phối target, descriptive statistics và các giá trị housing bất thường.


Output cho thấy điều gì?

Dataset gốc không có missing value và không có full duplicate rows, nhưng có repeated property `id` do một số nhà được bán nhiều lần. Đây là rủi ro leakage nếu cùng một property xuất hiện ở nhiều split, nên bước chuẩn bị sẽ giữ latest sale per `id` trước khi chia train/validation/test. Phân phối `price` lệch phải rõ rệt, vì vậy notebook dùng log1p target cho training và inverse transform khi báo metrics trên price scale gốc.


# 5. Data Preparation

Cell này làm gì?

Tạo feature engineering từ `date`, loại identifier `id`, chia train/validation/test theo price bins, fit preprocessing chỉ trên train, và quyết định target log-transform dựa trên skew.


Output cho thấy điều gì?

Preprocessing bắt đầu bằng cách bỏ full duplicates, loại `price <= 0` nếu có, và giữ latest sale per property `id` để tránh cùng một property đi qua nhiều split. Split được stratify theo price bins; preprocessor chỉ fit trên train set, còn validation/test chỉ dùng `transform`, nên scaling/encoding không học từ held-out data.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của dữ liệu deep learning và đánh giá CNN suitability cho housing-price tabular regression.


Output cho thấy điều gì?

`X_train` là ma trận `(samples, transformed features)`, trong đó zipcode đã được one-hot encode và numeric features đã scale. Đây là vector tabular; MLP regression là lựa chọn tự nhiên hơn CNN.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa Dense/Linear, ReLU, output regression, Conv1D và pooling bằng toy tensors.


Output cho thấy điều gì?

Regression output là một scalar tuyến tính, không phải xác suất hay class label. Conv1D/pooling chỉ minh họa khi dữ liệu có trục thứ tự; King County sau preprocessing không có cấu trúc cục bộ như vậy.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, forward network, output regression, MSE/MAE helper và Conv1D toy.


Output cho thấy điều gì?

NumPy forward pass minh họa cách MLP regression tạo một giá trị liên tục. MSE/MAE được tính trên target đã transform trong toy example; khi đánh giá model thật, notebook chuyển prediction về price scale gốc.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric regression và baseline Keras MLP với output scalar tuyến tính.


Output cho thấy điều gì?

Keras baseline nhận vector property đã transform và xuất một scalar linear trên target đã chuẩn hóa. Loss Huber phù hợp với target có skew/outliers vì bớt nhạy với sai số cực lớn hơn MSE thuần.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


Output cho thấy điều gì?

Training log hiển thị loss và MAE trên target đã transform cho train/validation. Checkpoint chọn epoch có validation loss thấp nhất; test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và MAE của baseline Keras trên train/validation, đồng thời lưu figure.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        def plot_history(history, title, output_path):
            history_df = pd.DataFrame(history.history)
            fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
            axes[0].plot(history_df['loss'], label='train')
            axes[0].plot(history_df['val_loss'], label='validation')
            axes[0].set_title(f'{title} Loss')
            axes[0].set_xlabel('Epoch')
            axes[0].legend()
            axes[1].plot(history_df['mae'], label='train')
            axes[1].plot(history_df['val_mae'], label='validation')
            axes[1].set_title(f'{title} MAE on transformed target')
            axes[1].set_xlabel('Epoch')
            axes[1].legend()
            fig.tight_layout()
            fig.savefig(output_path, dpi=160)
            plt.show()


Output cho thấy điều gì?

Curves cho biết model học ổn định hay overfit trên validation. Vì target đã log/scale, magnitude của loss dùng để chọn model; metric giá nhà dễ hiểu hơn được tính riêng sau khi inverse transform.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng MAE, MSE, RMSE, R²; lưu actual-vs-predicted, residual plot và error distribution.


Output cho thấy điều gì?

Keras baseline validation đạt MAE 70,219, RMSE 138,387, R² 0.8679. Diagnostic plots cho thấy sai số vẫn lớn hơn ở vùng giá cao, phù hợp với target có đuôi phải dài.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước train, chạy một batch qua model để kiểm tra shape và parameter count.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

        train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train_model.reshape(-1, 1), dtype=torch.float32))
        valid_ds = TensorDataset(torch.tensor(X_valid, dtype=torch.float32), torch.tensor(y_valid_model.reshape(-1, 1), dtype=torch.float32))
        test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test_model.reshape(-1, 1), dtype=torch.float32))

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

        class TorchMLPRegressor(nn.Module):
            def __init__(self, input_dim):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(input_dim, 128),
                    nn.ReLU(),
                    nn.Linear(128, 64),
                    nn.ReLU(),
                    nn.Linear(64, 1),
                )

            def forward(self, x):
                return self.net(x)

        def count_torch_params(model):
            return sum(p.numel() for p in model.parameters() if p.requires_grad)

        torch_model = TorchMLPRegressor(input_dim).to(DEVICE)
        features_batch, labels_batch = next(iter(train_loader))
        with torch.no_grad():
            output_batch = torch_model(features_batch.to(DEVICE))

        print("Input feature batch shape:", tuple(features_batch.shape))
        print("Regression output batch shape:", tuple(output_batch.shape))
        print("PyTorch parameter count:", count_torch_params(torch_model))


Output cho thấy điều gì?

Batch input có dạng `(batch, transformed features)` và output có dạng `(batch, 1)`, đúng với regression scalar. Kiến trúc PyTorch tương đương ý tưởng với Keras baseline để so sánh framework công bằng hơn.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline bằng loop thủ công, dùng SmoothL1Loss/Huber và lưu checkpoint tốt nhất theo validation loss.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        def torch_predict_scaled(model, loader):
            model.eval()
            preds = []
            targets = []
            with torch.no_grad():
                for xb, yb in loader:
                    xb = xb.to(DEVICE)
                    output = model(xb)
                    preds.append(output.cpu().numpy().ravel())
                    targets.append(yb.numpy().ravel())
            return np.concatenate(targets), np.concatenate(preds)

        criterion = nn.SmoothL1Loss()
        optimizer = torch.optim.Adam(torch_model.parameters(), lr=1e-3)

        torch_history_rows = []
        best_val_loss = np.inf
        best_state = None
        epochs_without_improvement = 0
        start = time.perf_counter()

        for epoch in range(1, MAX_EPOCHS + 1):
            torch_model.train()
            running_loss = 0.0
            for xb, yb in train_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                optimizer.zero_grad()
                output = torch_model(xb)
                loss = criterion(output, yb)
                loss.backward()
                optimizer.step()
                running_loss += float(loss.item()) * xb.size(0)

            train_loss = running_loss / len(train_loader.dataset)
            valid_scaled_true, valid_scaled_pred = torch_predict_scaled(torch_model, valid_loader)
            val_loss = float(np.mean(np.where(np.abs(valid_scaled_pred - valid_scaled_true) < 1, 0.5 * (valid_scaled_pred - valid_scaled_true) ** 2, np.abs(valid_scaled_pred - valid_scaled_true) - 0.5)))
            valid_price_pred_epoch = np.maximum(inverse_transform_target(valid_scaled_pred), 0)
            valid_metrics_epoch = regression_metrics(y_valid_np, valid_price_pred_epoch)
            torch_history_rows.append(
                {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "val_mae_price": valid_metrics_epoch["MAE"],
                    "val_rmse_price": valid_metrics_epoch["RMSE"],
                    "val_r2": valid_metrics_epoch["R2"],
                }
            )
            print(f"Epoch {epoch:02d} - train_loss={train_loss:.4f} - val_loss={val_loss:.4f} - val_rmse={valid_metrics_epoch['RMSE']:.2f} - val_r2={valid_metrics_epoch['R2']:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in torch_model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= PATIENCE:
                    print("Early stopping triggered.")
                    break

        torch_train_time = time.perf_counter() - start
        torch_model.load_state_dict(best_state)
        torch_checkpoint = MODEL_DIR / "pytorch_basic_mlp_regressor_best.pt"
        torch.save(
            {
                "model_state_dict": torch_model.state_dict(),
                "input_dim": input_dim,
                "feature_columns": feature_cols,
                "numeric_features": numeric_features,
                "categorical_features": categorical_features,
                "use_log_target": USE_LOG_TARGET,
                "target_center": target_center,
                "target_scale": target_scale,
                "seed": SEED,
            },
            torch_checkpoint,
        )
        torch_history = pd.DataFrame(torch_history_rows)
        torch_history.to_csv(METRIC_DIR / "pytorch_basic_mlp_history.csv", index=False)
        print(f"PyTorch train time: {torch_train_time:.2f} seconds")
        display(torch_history.tail())


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ `zero_grad`, forward, loss, backward và optimizer step. Validation RMSE/R² trong log đã inverse transform về price scale để dễ đọc hơn loss nội bộ.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng metric price-scale như Keras.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        valid_scaled_true, torch_valid_scaled_pred = torch_predict_scaled(torch_model, valid_loader)
        torch_val_pred = np.maximum(inverse_transform_target(torch_valid_scaled_pred), 0)
        torch_metrics = regression_metrics(y_valid_np, torch_val_pred)
        torch_result = {
            "Model": "M1 Basic MLP Regressor",
            "Framework": "PyTorch",
            **torch_metrics,
            "Params": int(count_torch_params(torch_model)),
            "Train Time": float(torch_train_time),
            "Checkpoint": torch_checkpoint.relative_to(PROJECT_ROOT).as_posix(),
        }
        save_json(METRIC_DIR / "pytorch_basic_mlp_validation_metrics.json", torch_result)

        display(pd.DataFrame([torch_result]).drop(columns=["Checkpoint"]).round(4))


Output cho thấy điều gì?

PyTorch baseline được đánh giá trên cùng validation split và cùng target inverse transform. Metric khác biệt giữa hai framework có thể đến từ initialization và training loop, không đủ để kết luận framework nào tốt hơn về bản chất.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng validation MAE/RMSE/R².


Output cho thấy điều gì?

Trong baseline framework comparison, PyTorch có validation RMSE thấp nhất (113,519) ở run này. Đây chỉ là kết quả một lần chạy với cùng ý tưởng architecture, không phải kết luận framework nào tốt hơn về bản chất.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture phù hợp dữ liệu tabular: M1 Basic MLP Regressor, M2 BatchNorm, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được train như production model vì feature order không có ý nghĩa không gian.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        def evaluate_keras_model(model, model_key, checkpoint_path, train_time):
            val_pred = predict_price_keras(model, X_valid)
            metrics = regression_metrics(y_valid_np, val_pred)
            result = {'Model': model.name.replace('_', ' '), 'Framework': 'Keras', **metrics, 'Params': int(model.count_params()), 'Train Time': float(train_time), 'Checkpoint': checkpoint_path.relative_to(PROJECT_ROOT).as_posix()}
            save_json(METRIC_DIR / f'{model_key}_validation_metrics.json', result)
            return result


Output cho thấy điều gì?

Các architecture đều giữ cùng split, preprocessing, target transform, seed và training budget. CNN được ghi nhận ở mức methodological investigation, không ép thành model vì bài toán là tabular regression.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation MAE/RMSE/R² và chọn ứng viên tốt nhất bằng validation RMSE thấp nhất.


Output cho thấy điều gì?

`M4 Residual MLP Feature Attention Regressor` là model tốt nhất trên validation với RMSE 119,459, MAE 67,889, R² 0.9015. Sau leakage fix repeated property `id`, M4 attention thắng validation RMSE nhưng vẫn cần cân nhắc vì parameter count cao hơn M1.


# 19. Representation Inspection

Cell này làm gì?

Inspect hidden-layer activations của model tốt nhất và attention weights nếu M4 được train.


Output cho thấy điều gì?

Hidden activations cho thấy model đã biến đổi vector property thành representation phi tuyến trước output giá. Attention weights, nếu dùng M4, là công cụ inspection model chứ không phải bằng chứng nhân quả về thị trường nhà.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi validation của model được chọn: absolute error lớn nhất, residual theo price range, và nhóm giá thấp/trung/cao.


In [ ]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    if RUN_TRAINING:
        range_rows = []


Output cho thấy điều gì?

Validation error lớn nhất nằm ở nhóm high-price với MAE khoảng 118,650 và RMSE khoảng 189,305. Điều này phù hợp với phân phối giá lệch phải: nhà giá cao tạo residual lớn hơn và làm RMSE nhạy hơn MAE.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation RMSE, evaluate test set đúng một lần, rồi lưu final model và diagnostics.


Output cho thấy điều gì?

Final test evaluation của `M4 Residual MLP Feature Attention Regressor` đạt MAE 70,282, RMSE 146,275, R² 0.8673 trên price scale gốc sau inverse transform. Test set chỉ dùng ở bước này sau khi chọn model bằng validation RMSE.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc artifacts A03 nếu có và so sánh dataset/workflow/metrics với A04. Dataset khác nhau nên không dùng score difference như benchmark trực tiếp.


Output cho thấy điều gì?

A03 và A04 cùng là housing-price regression nhưng khác thị trường, target scale và feature schema. Vì vậy chênh lệch RMSE/R² không chứng minh model A04 tốt hơn hay kém hơn A03; nó chỉ mô tả kết quả trên dataset A04.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


Output cho thấy điều gì?

Notebook đã lưu final model, preprocessor, comparison metrics, error-analysis CSV và figures. Kết quả cuối: selected model là `M4 Residual MLP Feature Attention Regressor`, test MAE 70,282, RMSE 146,275, R² 0.8673; tổng training/evaluation time được ghi nhận khoảng 82.39 giây.


## Kết quả gốc để đối chiếu

Các file đã lưu được đọc nguyên trạng; không fit hoặc forward TEST mới.

In [ ]:
import json
import pandas as pd
from IPython.display import display
for _relative in ['A04/artifacts/pipeline04/cifar10/metrics/all_model_comparison.csv', 'A04/artifacts/pipeline04/cifar10/metrics/architecture_comparison.csv', 'A04/artifacts/pipeline04/cifar10/metrics/framework_comparison.csv', 'A04/artifacts/pipeline04/cifar10/metrics/selection_comparison.csv', 'A04/artifacts/pipeline04/customer_behavior/metrics/a03_vs_a04_comparison.csv', 'A04/artifacts/pipeline04/customer_behavior/metrics/architecture_comparison.csv', 'A04/artifacts/pipeline04/customer_behavior/metrics/keras_vs_pytorch_comparison.csv', 'A04/artifacts/pipeline04/diabetes/metrics/a03_vs_a04_comparison.csv', 'A04/artifacts/pipeline04/diabetes/metrics/architecture_comparison.csv', 'A04/artifacts/pipeline04/diabetes/metrics/keras_vs_pytorch_comparison.csv', 'A04/artifacts/pipeline04/house_price/metrics/a03_vs_a04_comparison.csv', 'A04/artifacts/pipeline04/house_price/metrics/architecture_comparison.csv']:
    _path = REPO / _relative
    print(_relative)
    if _path.suffix == ".csv": display(pd.read_csv(_path))
    else: display(json.loads(_path.read_text()))


In [ ]:
import inspect
_notebook_defined = [(name, key) for name, mod in INLINE_MODULES.items() for key, obj in vars(mod).items() if inspect.isfunction(obj) and obj.__module__ == name]
assert all(vars(INLINE_MODULES[name])[key].__code__.co_filename.startswith("<notebook>/") for name,key in _notebook_defined)
print({"module có code ngay trong notebook": sum(bool(getattr(m,"__notebook_source__",False)) for m in INLINE_MODULES.values()), "hàm inline":len(_notebook_defined), "train/TEST đã chạy":False})
